# Transformer Foundations, Part 3 of 3: Encoder-Decoder and Cross-Attention

> **Where you are:** [Part 1](01-attention-and-transformer-blocks.ipynb) built the common block. [Part 2](02-decoder-only-language-model.ipynb) trained one causal stack. Part 3 asks what changes when the source deserves a full bidirectional reading before generation begins.

The detailed integer-reversal chapter remains the main path. An optional appendix near the end preserves the original compact `the cat sat on the mat` architecture-family lab from the former all-in-one Transformer notebook.


> **History.** Sutskever, Vinyals, and Le (Google, 2014) showed that a fixed-size vector — the final hidden state of an RNN encoder — could represent an entire sentence well enough for machine translation. The bottleneck was the vector size. Bahdanau, Cho, and Bengio (2015) introduced additive attention: instead of one fixed vector, the decoder could read a weighted blend of all encoder hidden states. This was the direct ancestor of the Transformer's cross-attention mechanism. The encoder-decoder pattern still powers T5, BART, mT5, and every sequence-to-sequence production model today.
>
> **Where you are.** You've built a full Transformer in `02-transformers/` — you understand self-attention, positional encoding, residuals, and multi-head attention. The gap: you built decoder-only (MiniLM) and briefly saw the encoder-only variant. You have not yet trained the encoder-decoder architecture — the variant that handles tasks where input and output sequences differ in length and meaning.
>
> **Notation.** $S$ — source sequence length; $T$ — target sequence length; $d_{model}$ — embedding dimension; $Q$ from decoder, $K$/$V$ from encoder in cross-attention; `mask=None` for encoder (bidirectional), causal mask for decoder; teacher forcing: feed ground-truth target at each decoding step during training.


---

## Prerequisite Bridge — From `02-transformers/01-attention-and-transformer-blocks.ipynb`

| Foundation                                              | Role in this notebook                                                                                                                                |
| ------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------- |
| Multi-head self-attention (Q, K, V, scaled dot-product) | Used directly as `MultiHeadSelfAttention` in both encoder and decoder blocks — no re-derivation                                                      |
| Causal attention mask (triangular)                      | The decoder's causal self-attention uses this exact mask; the encoder deliberately removes it                                                        |
| Sinusoidal positional encoding                          | Carried forward unchanged into the encoder and decoder embeddings                                                                                    |
| Residual connections + LayerNorm (Pre-LN)               | Both blocks follow the same Pre-LN pattern built in `02-transformers`                                                                                |
| Decoder-only architecture (`MiniLM`)                    | This notebook extends that to encoder + cross-attention; the decoder block here is the same decoder-only block plus one new cross-attention sublayer |

> **If you haven't run `02-transformers/01-attention-and-transformer-blocks.ipynb`** the attention mechanism, causal masking, and Pre-LN architecture used here will not be familiar — those derivations are not repeated. Complete that notebook first.


## 0 · The Challenge

> **The mission**: Build an encoder-decoder Transformer from scratch and prove it learns a non-trivial mapping — integer sequence reversal (`[3, 1, 4, 1] → [1, 4, 1, 3]`).

**What we know so far:**

- Decoder-only Transformers (MiniLM) generate text left-to-right from context.
- The causal mask prevents the decoder from seeing future tokens.
- **But we still can't**: map a variable-length source sequence to a different-length target sequence, or give the decoder access to the full source representation at every decoding step.

**What's blocking us:**
A decoder-only model reads its own previous output. It cannot read a separately encoded source. The bottleneck: a single fixed-size vector (the last encoder hidden state) loses positional specificity — cosine similarity between encoder outputs collapses toward the mean.

**What this chapter unlocks:**
Cross-attention — the decoder queries the encoder's full output at every step. Every source token's representation is available to every decoding step. Held-out sequence accuracy will test whether the model learned reversal; the trained cross-attention map will diagnose whether its routing matches that rule.


## Encoder-Decoder Transformers in PyTorch

## From Sequence Reversal to seq2seq Translation

This notebook builds an encoder-decoder transformer from first principles in PyTorch, using a single running example — reversing a short sequence of integers — to make every moving part observable before bridging to T5/BART.

| Step | Part                  | Concept                                   | Key Idea                                                      |
| ---- | --------------------- | ----------------------------------------- | ------------------------------------------------------------- |
| 1    | The Contract          | What encoder-decoder solves               | Variable-length I/O + bidirectionality                        |
| 2    | The Encoder           | Bidirectional attention                   | Every token sees every other token                            |
| 3    | The Bottleneck        | Why naive concat fails                    | Fixed vector cannot hold a long sequence                      |
| 4    | Cross-Attention       | The bridge                                | Q = decoder, K = V = encoder                                  |
| 5    | Full Model + Training | Wire encoder + cross-attn + decoder       | Train on sequence reversal                                    |
| 6    | Cross-Attention Map   | Visualise what was learned                | Decoder step $i$ attends to source position $n-i$             |
| 6a   | Free-Running Decoding | Teacher forcing vs. autoregressive greedy | Exposure bias measured directly (+ a short beam-search aside) |
| 7    | Toy to Real           | T5 / BART parameter mapping               | Same architecture, wider vectors                              |


## The Encoder-Decoder Contract at a Glance

![Encoder-decoder sequence-to-sequence contract: bidirectional source encoding, cross-attention, and causal target generation](images/encoder-decoder-contract.png)

The encoder produces a contextual representation for each source token. The decoder generates the target sequence one step at a time, using causal self-attention over earlier target tokens and cross-attention over the complete encoder output.


The full encode-decode pipeline at inference time:

```mermaid
graph LR
    Src[Source tokens] --> EmbEnc[Embed + PE]
    EmbEnc --> Enc["Encoder\n(N × bidirectional SA → FFN)"]
    Enc --> Ctx["Context vectors\n(one per source token)"]

    Tgt[Target tokens so far] --> EmbDec[Embed + PE]
    EmbDec --> DecSA["Decoder\n(causal SA)"]
    DecSA --> CrossAttn["Cross-Attention\nQ ← decoder, K/V ← Context"]
    Ctx --> CrossAttn
    CrossAttn --> FFN[FFN → lm_head]
    FFN --> Out[Next target token]
```

Key: the encoder runs **once** over the full source; the decoder runs **step-by-step**, querying the fixed context vectors at each position via cross-attention.


> **PyTorch → Keras:** `torch.manual_seed(42)` fixes PyTorch's global RNG for reproducible weight init and dropout; `torch.device("cpu")` explicitly pins tensors/modules to a device (call `.to(device)` later to move them). **Keras/TF equivalent:** `tf.random.set_seed(42)` seeds TensorFlow's global RNG (Keras layers read from it at build time); device placement is usually implicit (`tf.config.list_physical_devices` + automatic GPU/CPU placement) rather than an explicit `device` object passed to every module call.

In [ ]:
#  Setup: imports and reproducibility seed
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

# Fix every RNG source so all numbers below are reproducible across reruns
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)
print("Seed fixed at 42 — every cell in this notebook is deterministic.")
print("  -> Re-running any cell produces the same numbers as in the prose above it.")

---

## Part 1 — The Encoder-Decoder Contract

### What problem does it solve that decoder-only cannot?

A decoder-only model (GPT-style) produces one token at a time, conditioned on every
token that came before in a single flat sequence. It is excellent at language modelling
— but consider **sequence reversal**: input `[3, 1, 4, 1]` must produce `[1, 4, 1, 3]`.
The first output token (`1`) depends on the _last_ input token (`1` at position 3).
A causal decoder, at generation step 0, cannot look forward to see position 3.

The 2x2 taxonomy of sequence-to-sequence tasks:

|                             | Same vocabulary / same length   | Variable-length output                              |
| --------------------------- | ------------------------------- | --------------------------------------------------- |
| **Unidirectional (causal)** | Language modelling (GPT, LLaMA) | Hard: future source tokens unavailable              |
| **Bidirectional**           | Sentence classification (BERT)  | **Encoder-Decoder**: T5, BART, original Transformer |

**The encoder-decoder contract:**

- The **encoder** reads the full source sequence in both directions and produces one
  enriched context vector per source token. It does not generate; it _enriches_.
- The **decoder** generates the target sequence autoregressively, querying the full
  source map via cross-attention at every step.

The cross-attention formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

where $Q$ comes from the **decoder** state and $K, V$ come from the **encoder** output.
The $\sqrt{d_k}$ scaling prevents dot-products from growing so large that softmax
saturates — a problem we will measure in Part 2.


In [ ]:
#  Running example: integer sequence reversal
#
# Toy vocabulary: digits 0-9 plus three special tokens.
# Task: reverse a length-4 sequence, e.g. [3, 1, 4, 1] -> [1, 4, 1, 3].
# This forces cross-attention to learn a non-trivial src->tgt routing:
#   output position 0 must attend to source position 3, etc.

PAD, BOS, EOS = 10, 11, 12
VOCAB_SIZE = 13  # 0-9 digits + PAD + BOS + EOS
SEQ_LEN = 4  # source / target sequence length
D_MODEL = 32  # embedding / hidden dimension (tiny for visibility)
N_HEADS = 4
D_FF = 64
N_LAYERS = 2


def make_reversal_pairs(n_samples, seq_len=SEQ_LEN, seed=42):
    """Generate (src, tgt) pairs where tgt = list(reversed(src))."""
    rng = np.random.default_rng(seed)
    srcs, tgts = [], []

    # Draw n_samples independent random sequences and reverse each for the target
    for _ in range(n_samples):
        src = list(rng.integers(0, 10, size=seq_len))
        tgt = src[::-1]
        srcs.append(src)
        tgts.append(tgt)
    return srcs, tgts


train_srcs, train_tgts = make_reversal_pairs(2000)
val_srcs, val_tgts = make_reversal_pairs(200, seed=99)

print("Reversal task examples (first 5):")
print(f"  {'Source':<20}  Target (reversed)")
print(f"  {'-'*18}  {'-'*18}")
for s, t in zip(train_srcs[:5], train_tgts[:5]):
    print(f"  {str(s):<20}  {str(t)}")
print()
print(f"Training pairs : {len(train_srcs)}")
print(f"Vocab size     : {VOCAB_SIZE}  (0-9 = digits, 10=PAD, 11=BOS, 12=EOS)")
print("Decoder input  : [BOS] + target[:-1]  (teacher forcing)")
print("Decoder target : target + [EOS]")

---

## Part 2 — The Encoder: Enriching Source Representations

### Bidirectional self-attention

In a **decoder** block, position $i$ can attend only to positions $0, \ldots, i$.
This is enforced by adding $-\infty$ to the upper-triangle of the score matrix before
the softmax. In an **encoder** block, we simply omit that mask. Every token sees
every other token — backward _and_ forward — in the very first block.

The complete implementation difference between an encoder and a decoder block is one
argument:

```
decoder block:  self_attn(x, mask=causal_mask)  # upper-triangle -> -inf
encoder block:  self_attn(x, mask=None)          # nothing blocked
```

Everything else — `MultiHeadSelfAttention`, `LayerNorm`, `FeedForward` — is identical.
We will confirm this with the heatmap experiment below.


### How the Encoder Builds Source Representations

![Encoder pipeline from source embeddings and positional information through bidirectional self-attention to one contextual representation per source token](images/encoder-source-representations.png)

The encoder produces one contextual vector for every source position, rather than a single fixed summary. Its self-attention is bidirectional, so each source token can use information from every other source token.


> **PyTorch → Keras:** `nn.Module` + `nn.Linear(d_model, d_model, bias=False)` defines learnable projections explicitly inside `__init__`, and `forward()` manually computes `Q @ K.transpose(-2,-1) / sqrt(d_k)`, applies `masked_fill(mask, -inf)`, then `F.softmax(...)`. **Keras/TF equivalent:** `tf.keras.layers.MultiHeadAttention(num_heads, key_dim)` (or subclassing `tf.keras.layers.Layer` with `self.add_weight`/`Dense` sublayers built in `build()`) provides the same scaled dot-product attention and masking (`attention_mask` argument) as a single built-in call, rather than the hand-rolled projection/reshape/softmax shown here.

In [ ]:
#  MultiHeadSelfAttention: the shared attention primitive
#
# Variable names mirror the math:
#   W_Q, W_K, W_V  projection matrices
#   Q, K, V        projected queries, keys, values
#   d_k            per-head dimension  (d_model // n_heads)
# The optional `mask` argument is the ONLY thing that will later distinguish
# an encoder block (mask=None) from a decoder block (mask=causal_mask).


class MultiHeadSelfAttention(nn.Module):
    """Multi-head self-attention with optional causal mask."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, S, D = x.shape

        # Project and reshape to (B, n_heads, S, d_k)
        Q = self.W_Q(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product: softmax( Q K^T / sqrt(d_k) ) V
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:

            # mask: (S, S) bool; True = blocked position
            scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))
        attn_w = F.softmax(scores, dim=-1)  # (B, n_heads, S, S)
        out = torch.matmul(attn_w, V)  # (B, n_heads, S, d_k)

        # Merge heads back into a single (B, S, D) tensor
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.W_O(out), attn_w

Every attention sublayer is paired with a position-wise feed-forward network — a
small 2-layer MLP applied independently to each token's vector, giving the model
nonlinear capacity that attention (a weighted average) cannot provide alone.


> **PyTorch → Keras:** `nn.Linear(d_model, d_ff)` + `F.relu(...)` + `nn.Linear(d_ff, d_model)` defines the two-layer position-wise MLP as explicit submodules wired together in `forward()`. **Keras/TF equivalent:** `tf.keras.Sequential([Dense(d_ff, activation="relu"), Dense(d_model)])` expresses the same sublayer declaratively — Keras `Dense` layers infer their input dimension automatically on first call, whereas `nn.Linear` requires the input size up front.

In [ ]:
#  FeedForward: position-wise sublayer applied after every attention block
class FeedForward(nn.Module):
    """Position-wise FFN: Linear(d_model -> d_ff) + ReLU + Linear(d_ff -> d_model)."""

    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

Combine the attention primitive above with a feed-forward sublayer and residual
connections into one `EncoderBlock`. The `mask=None` passed to `self.attn` is the
single line that makes this block bidirectional rather than causal.


> **PyTorch → Keras:** `nn.LayerNorm(d_model)` normalises the last dimension per-token; `forward()` manually implements the pre-norm residual pattern (`x = x + self.attn(self.norm1(x))`) by adding tensors directly. **Keras/TF equivalent:** `tf.keras.layers.LayerNormalization()` is the direct analogue, but residual wiring is normally expressed with the Functional API (`x = layers.Add()([x, sublayer(norm(x))])`) rather than Python `+` inside a subclassed layer's `call()`.

In [ ]:
#  EncoderBlock: attention + FFN with residual connections
class EncoderBlock(nn.Module):
    """Encoder layer: bidirectional self-attn (mask=None) + FFN, both with residual."""

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x):

        # Pre-norm residual connections (GPT-2 / T5 style)
        attn_out, attn_w = self.attn(
            self.norm1(x), mask=None
        )  # <- mask=None is the key
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, attn_w

Encoder blocks alone don't know token order — attention treats a sequence as an
unordered set of vectors. Add fixed sinusoidal position encodings to the token
embeddings, then stack `n_layers` `EncoderBlock`s into the full `MiniEncoder`.


> **PyTorch → Keras:** `nn.Embedding(vocab_size, d_model, padding_idx=PAD)` is a lookup table that also zeroes the gradient for the PAD row; `register_buffer("pe", ...)` stores the fixed (non-trainable) sinusoidal table so it moves with `.to(device)` but is excluded from `optimizer.parameters()`; `nn.ModuleList([...])` holds a variable number of sub-blocks that PyTorch auto-registers as children. **Keras/TF equivalent:** `tf.keras.layers.Embedding(vocab_size, d_model, mask_zero=True)` handles padding via masking rather than a zeroed embedding row; a fixed positional table is usually a plain `tf.constant` attribute (no buffer/parameter distinction exists in Keras); a list of sublayers is just a Python list of layers assigned as an attribute — Keras tracks them automatically too.

In [ ]:
#  sinusoidal_pe + MiniEncoder: position encodings and the full stack
def sinusoidal_pe(max_seq, d_model):
    """Return sinusoidal positional encoding of shape (max_seq, d_model)."""
    pe = torch.zeros(max_seq, d_model)
    pos = torch.arange(0, max_seq, dtype=torch.float).unsqueeze(1)

    # Frequencies spaced geometrically across dimensions: 1 / 10000^(2i/d_model)
    div = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
    )

    # Even dimensions get sin, odd dimensions get cos, at each position
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


class MiniEncoder(nn.Module):
    """
    Bidirectional encoder (BERT / T5-encoder style).
    Output: one enriched d_model-dimensional vector per source token.
    NOT a next-token predictor — a context enricher.
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)

        # Precompute fixed (non-trainable) positional encodings once
        self.register_buffer("pe", sinusoidal_pe(max_seq, d_model))

        # Stack n_layers independent encoder blocks
        self.blocks = nn.ModuleList(
            [EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )
        self.norm_out = nn.LayerNorm(d_model)

    def forward(self, token_ids):
        S = token_ids.shape[1]

        # Add positional info to token embeddings (trim PE to actual sequence length)
        x = self.token_emb(token_ids) + self.pe[:S]
        all_attn = []

        # Run every encoder block in sequence, collecting attention weights for inspection
        for block in self.blocks:
            x, aw = block(x)
            all_attn.append(aw)
        return self.norm_out(x), all_attn

With `MiniEncoder` assembled, sanity-check its output shapes on our running example
`[3, 1, 4, 1]` before trusting it anywhere else in this notebook.


> **PyTorch → Keras:** `torch.tensor([[3, 1, 4, 1]])` builds an integer input tensor and `enc_test(dummy_ids)` calls `forward()` implicitly via `nn.Module.__call__`. **Keras/TF equivalent:** `tf.constant([[3, 1, 4, 1]])` builds the equivalent tensor, and `enc_test(dummy_ids)` likewise invokes a subclassed `Layer`/`Model`'s `call()` — the calling convention is nearly identical between the two frameworks.

In [ ]:
#  Sanity check
# Re-seed so this check reproduces the same encoder weights every run
torch.manual_seed(42)
enc_test = MiniEncoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
dummy_ids = torch.tensor([[3, 1, 4, 1]])

# Forward pass on the running example to confirm output shapes match expectations
enc_out, enc_attns = enc_test(dummy_ids)

print("Encoder output shape:", tuple(enc_out.shape))
print(f"  -> (batch=1, seq_len={SEQ_LEN}, d_model={D_MODEL})")
print("  -> One enriched vector per source token, NOT a next-token prediction")
print()
print("Attention weight shape per block:", tuple(enc_attns[0].shape))
print(f"  -> (batch=1, n_heads={N_HEADS}, src={SEQ_LEN}, src={SEQ_LEN})")
print("  -> Every source token can attend to every other source token (no mask)")

### Code Walkthrough: MiniEncoder Building Blocks, Recapped

**The pieces above, split into small cells — 4 key patterns worth re-emphasizing:**

---

**`W_Q(x).view(B, S, n_heads, d_k).transpose(1, 2)` — split heads for parallel attention**
Each token's `d_model`-dimensional embedding is projected to Q/K/V then split into `n_heads` independent sub-vectors of size `d_k = d_model // n_heads`. The `.transpose(1, 2)` swaps sequence and head axes, giving shape `(B, n_heads, S, d_k)`. Each head independently attends over the full sequence but works in its own lower-dimensional subspace.

---

**`mask.unsqueeze(0).unsqueeze(0)` — broadcast the mask over batch and head dims**
The raw mask has shape `(S, S)`. Two `.unsqueeze(0)` calls prepend a batch dim and a head dim, yielding `(1, 1, S, S)`. PyTorch broadcasts this over all batches and all heads without allocating extra memory — the same pattern appears in every production GPT/BERT implementation. Passing `mask=None` in `EncoderBlock` skips this entirely, giving bidirectional (unrestricted) attention.

---

**`sinusoidal_pe(max_seq, d_model)` — fixed positional fingerprints added to embeddings**
Even-indexed dimensions use $\sin(pos / 10000^{2i/d})$ and odd dimensions use $\cos(\ldots)$. Different-frequency sinusoids produce a unique fingerprint for every position that never repeats. Adding PE to token embeddings lets the model see _both_ what a token is and where it sits — without any trainable parameters.

---

**`EncoderBlock` with `mask=None` — bidirectional self-attention is the encoder's superpower**
`self.attn(self.norm1(x), mask=None)` removes the causal restriction. Token at position 0 directly attends to position 99 in a single layer, and vice versa. This is the key difference from a decoder: the encoder builds a rich **context map** for every source position, not a next-token predictor.

> **PyTorch shape note:** The encoder output is `(B, seq_len, d_model)` — one enriched vector _per source token_, not a single sequence vector. Cross-attention later queries these per-position vectors individually, which is why the decoder can attend to any source position at each generation step.


#### Predict before you run — encoder heatmap row 0

In the decoder's causal heatmap, row 0 has exactly **1** non-zero cell (token 0
attends only to itself — it cannot see the future).

**Predict:** In the encoder heatmap, how many non-zero cells will row 0 have?

A. 1 (same as decoder — only itself)
B. 2 (attends to immediate neighbours only)
C. 4 (attends to all positions equally)

Write your answer, then run the encoder-vs-decoder heatmap experiment below to see the reveal.


> **PyTorch → Keras:** `torch.randn(1, SEQ_LEN, D_MODEL)` samples a random input tensor; `torch.triu(torch.ones(S, S, dtype=torch.bool), diagonal=1)` builds a boolean upper-triangular mask consumed directly by the hand-rolled `masked_fill` inside `MultiHeadSelfAttention`. **Keras/TF equivalent:** `tf.random.normal(...)` is the RNG analogue; for masking, `tf.keras.layers.MultiHeadAttention` takes a boolean `attention_mask` of shape `(..., T, S)` where `True` means *attend* (the opposite convention from this notebook's "True = blocked"), so a manually built causal mask must be inverted before passing it to a Keras `MultiHeadAttention` layer.

In [ ]:
#  Encoder vs Decoder attention heatmap: SAME weights, ONLY the mask differs
#
# By running the same MultiHeadSelfAttention layer twice — once with mask=None
# and once with a causal mask — we isolate the mask as the single causal variable.

torch.manual_seed(7)
shared_mha = MultiHeadSelfAttention(D_MODEL, N_HEADS)

x_demo = torch.randn(1, SEQ_LEN, D_MODEL)  # same input for both runs

# Upper-triangle mask: True = blocked (sent to -inf before softmax)
causal_mask = torch.triu(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool), diagonal=1)

_, w_encoder = shared_mha(x_demo, mask=None)  # bidirectional
_, w_decoder = shared_mha(x_demo, mask=causal_mask)  # causal

# Head 0 for display
w_enc_h0 = w_encoder[0, 0].detach().numpy()
w_dec_h0 = w_decoder[0, 0].detach().numpy()
labels = ["3", "1", "4", "1"]  # our running example token values

# Draw encoder (bidirectional) and decoder (causal) attention as side-by-side heatmaps
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, data, title, cmap in [
    (axes[0], w_enc_h0, "Encoder (mask=None)\nBidirectional", "Blues"),
    (axes[1], w_dec_h0, "Decoder (causal mask)\nLower-triangle only", "Oranges"),
]:

    # cbar=True (the default): a continuous-magnitude heatmap needs its own
    # colorbar as the legend for what a color means — annot=True alone is not
    # a substitute once the reader compares two differently-colored panels.
    sns.heatmap(
        data,
        ax=ax,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        xticklabels=labels,
        yticklabels=labels,
        linewidths=0.5,
        cbar=True,
        vmin=0,
        vmax=1,
        cbar_kws={"label": "attention weight"},
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    ax.tick_params(axis="x", rotation=0)

plt.suptitle(
    "Same MHA weights, same input — only the mask differs",
    fontsize=11,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# Count non-negligible attention weights in row 0 for encoder vs. decoder
nz_enc = int((w_enc_h0[0] > 0.01).sum())
nz_dec = int((w_dec_h0[0] > 0.01).sum())
print(f"Row 0 non-zero cells — Encoder: {nz_enc}   Decoder: {nz_dec}")
print(f"  -> Encoder row 0 attends to ALL {SEQ_LEN} positions  (answer: C)")
print(f"  -> Decoder row 0 attends to only 1 position (cannot see the future)")
print()
print("Implementation difference: one argument — mask=None vs mask=causal_mask.")
print("  -> Every other component (MHA, FFN, LayerNorm) is shared code.")

#### What just happened — and what is missing

We proved that `mask=None` gives every token a 360-degree view of the source.
Token `4` at position 2 can incorporate signals from `3` (position 0) and `1`
(position 3) in the very first block.

### Why a Full-Sentence View Changes Meaning

Now replace the digits with two source sentences:

```text
Working at a [MASK]
Washing clothes at a [MASK]
```

The token at the final position is `[MASK]` in both sentences. It does **not** begin as an empty vector: `[MASK]` is a learned vocabulary token with its own embedding, plus positional information. What changes is the context blended into that position by bidirectional self-attention.

```text
same [MASK] embedding              same [MASK] embedding
          +                                 +
Working / at / a context           Washing / clothes / at / a context
          ↓                                 ↓
employment/location features       cleaning/fabric/location features
          ↓                                 ↓
bank or office becomes plausible   laundromat or dry cleaner becomes plausible
```

The encoder therefore returns a **contextual representation** at the masked position, not a dictionary definition of `[MASK]`. In a masked-language-model head, that representation can be projected to a vocabulary distribution. The representation is also useful without a mask: every ordinary source token is enriched by both its left and right context before another component consumes it.

> **Diagnostic, not proof:** an attention map can show which positions received weight in one head, but a large weight alone does not prove that a token caused the prediction. The final representation also depends on values, other heads, residual paths, feed-forward layers, and all preceding layers.

We now have a box of enriched source vectors. The question is: how does the decoder
consume them? The naive approach — concatenate encoder output to the decoder context —
has a hidden flaw. Part 3 exposes it.


#### Your turn — how many heads make attention patterns diverge?

Part 2 built `MultiHeadSelfAttention` with `N_HEADS=4`. Each head learns its own
`W_Q`/`W_K`/`W_V` and therefore its own attention pattern over the same input.

**Predict:** with only **1** head, is there anything to compare (a single pattern is
trivially identical to itself)? With **4** or **8** heads, will the patterns be nearly
identical (redundant) or clearly different (specialised)?

Change `ex_n_heads` below and re-run to see how head count changes the diversity of
attention patterns computed on our running example `[3, 1, 4, 1]`.


### Why Cross-Attention Replaces a Fixed Context Vector

![Historical fixed-context bottleneck compared with a Transformer encoder-decoder using cross-attention over all encoder outputs](images/fixed-context-bottleneck.png)

Earlier sequence-to-sequence models often compressed the source into one fixed-size vector. Cross-attention lets each decoding step consult the full set of encoder representations instead, so different target positions can focus on different source tokens.


> **PyTorch → Keras:** `mha_ex(x_ex, mask=None)` re-runs the same hand-written attention module with a different head count, and `.detach().numpy()` pulls the attention weights out of the autograd graph into plain NumPy for the correlation check. **Keras/TF equivalent:** `tf.keras.layers.MultiHeadAttention(num_heads=ex_n_heads, key_dim=...)` exposes attention scores via `return_attention_scores=True`; converting to NumPy needs no `.detach()` step since eager TF tensors convert directly with `.numpy()` (no separate autograd graph to detach from).

In [ ]:
#  EXERCISE — head-count vs. attention-pattern diversity
# Change ex_n_heads (must divide D_MODEL=32: try 1, 2, 4, 8) and predict whether
# more heads produce more DIFFERENT attention patterns on our running example.

ex_n_heads = 4  # CHANGE: try 1, then 2, then 8

torch.manual_seed(42)

# Build a fresh MHA layer with the chosen head count
mha_ex = MultiHeadSelfAttention(D_MODEL, ex_n_heads)
x_ex = torch.randn(1, SEQ_LEN, D_MODEL)
_, w_ex = mha_ex(x_ex, mask=None)  # bidirectional, same as MiniEncoder

# Flatten each head's attention matrix into one row for correlation comparison
patterns = w_ex[0].detach().reshape(ex_n_heads, -1).numpy()
print(f"{ex_n_heads} head(s) on our running example [3, 1, 4, 1]:")

# With >1 head, measure how correlated (similar) the heads' attention patterns are
if ex_n_heads > 1:
    corrs = np.corrcoef(patterns)

    # Exclude the diagonal (each head trivially correlates with itself)
    off_diag = corrs[~np.eye(ex_n_heads, dtype=bool)]
    print(
        f"  Pairwise correlation between head attention patterns: mean={off_diag.mean():.3f}"
    )
    if off_diag.mean() < 0.5:
        print("  -> Heads show CLEARLY DIFFERENT attention patterns (specialised).")
    else:
        print("  -> Heads are still fairly correlated at this random init (untrained).")
else:
    print(
        "  -> Only 1 head exists — nothing to compare against, correlation undefined."
    )

print(
    f"  -> Each head works in a {D_MODEL // ex_n_heads}-dim subspace "
    f"(d_k = d_model / n_heads = {D_MODEL}/{ex_n_heads})."
)

---

## Part 3 — The Bottleneck Problem

### Why passing encoder output as decoder initial state fails

Pre-attention seq2seq models (Sutskever et al., 2014) compressed the entire source
sequence into a single fixed-size vector — the last hidden state of an RNN encoder —
and handed it to the decoder as its initial hidden state. This is the **information
bottleneck**.

The capacity of a $d$-dimensional vector is fixed regardless of source length:

$$\text{bottleneck capacity} \propto d_{\text{model}}$$

For a 3-word sentence that capacity might suffice. For a 50-word paragraph it does not —
the vector must carry everything the decoder will ever need, and accuracy collapses on
anything beyond ~30 words. This was a fundamental limitation of early seq2seq systems.

Cross-attention eliminates the bottleneck by keeping **all** $S$ source vectors
simultaneously accessible:

$$\text{cross-attention capacity} \propto d_{\text{model}} \times S_{\text{src}}$$

For $S_{\text{src}} = 50$ tokens that is $50\times$ the information access of the
single-vector approach.


#### Predict before you run — does mean-pooling preserve positional identity?

We are about to measure cosine similarity between per-position encoder vectors and
mean-pooled encoder vectors for two very different source sequences:
`[3, 1, 4, 1]` vs `[9, 8, 7, 6]`.

**Predict:** After mean-pooling the encoder output, will the two sequences be:

A. Clearly distinct (cosine similarity < 0.3) — mean pool retains full positional info
B. Moderately similar (0.3 to 0.7) — some information lost
C. Very similar (> 0.7) — pooling collapses positional differences

Write your answer, then run the cosine-similarity experiment below to find out.


> **PyTorch → Keras:** `enc_probe.eval()` switches off training-only behaviour (dropout/batchnorm running stats) and `torch.no_grad()` disables autograd graph-building for the forward passes that follow. **Keras/TF equivalent:** Keras models are called with `training=False` (e.g. `enc_probe(seq_a, training=False)`) instead of a persistent `.eval()` mode switch, and there is no `no_grad()` equivalent needed outside a `tf.GradientTape()` block — plain forward calls never build a gradient tape unless you explicitly open one.

In [ ]:
#  Prove the bottleneck: mean-pooled encoder vector loses positional detail
#
# Strategy: compare two different source sequences.
# Per-position encoder outputs should be clearly distinct (different content).
# Mean-pooled encoder output collapses them toward each other.
#
# We measure cosine similarity between seq_a and seq_b at each encoder position
# and compare to the mean-pooled cosine similarity.

torch.manual_seed(42)

# Fresh encoder instance used only as a probe for this comparison
enc_probe = MiniEncoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
enc_probe.eval()

seq_a = torch.tensor([[3, 1, 4, 1]])  # our main example
seq_b = torch.tensor([[9, 8, 7, 6]])  # completely different sequence

# No gradients needed — this is inference only, for measurement
with torch.no_grad():
    out_a, _ = enc_probe(seq_a)  # (1, 4, 32)
    out_b, _ = enc_probe(seq_b)  # (1, 4, 32)


# Cosine similarity between two vectors (normalize then dot product)
def cos_sim(a, b):
    a = a / (a.norm() + 1e-9)
    b = b / (b.norm() + 1e-9)
    return float((a * b).sum())


# Compare seq_a vs seq_b at each individual encoder position
pos_sims = [cos_sim(out_a[0, i], out_b[0, i]) for i in range(SEQ_LEN)]

# Compare the same two sequences after mean-pooling across positions
pool_sim = cos_sim(out_a.mean(dim=1)[0], out_b.mean(dim=1)[0])

print("Cosine similarity: seq_a=[3,1,4,1] vs seq_b=[9,8,7,6]")
print()
print("Per-position encoder vectors:")
for i, s in enumerate(pos_sims):
    bar = "#" * int(abs(s) * 20)
    tag = "distinct" if abs(s) < 0.7 else "similar"
    print(f"  position {i}: {s:+.4f}  {bar}  ({tag})")
print()
print(f"Mean-pooled vector: {pool_sim:+.4f}")
print()
print("  -> Individual encoder positions encode content distinctly.")
print("  -> Mean-pooling dilutes positional specificity.")
print(f"  -> With longer sequences (S=50) the mean pool becomes a 'blur'.")
print()
print("Cross-attention keeps all", SEQ_LEN, "source vectors alive.")
print(
    "  -> The decoder queries exactly the positions it needs at each generation step."
)

#### What just happened — and what is missing

We measured that mean-pooling the encoder output loses per-position distinction.
The fix is to keep all $S$ source vectors available — and let the decoder dynamically
choose which ones to attend to at each generation step. That dynamic query mechanism
is cross-attention. Part 4 builds it.


---

## Part 4 — Cross-Attention: The Bridge

### Two problems with the naive alternatives

Part 3 proved one naive alternative — compressing the source into a single fixed
vector — loses positional detail. There is a second naive alternative worth ruling
out before introducing cross-attention: _why not just feed the encoder's per-position
vectors into the decoder as extra tokens in its own self-attention sequence?_

**Problem 1 — The bottleneck (recap from Part 3).** A single pooled vector cannot
carry $S$ positions' worth of distinct content once $S$ grows past a handful of
tokens — we measured this directly with cosine similarity above.

**Problem 2 — Concatenation into self-attention breaks on two counts.** Even if we
tried to sidestep the bottleneck by prepending the encoder's $S$ enriched vectors to
the decoder's own sequence and running one shared self-attention over all of it:

- The decoder's **causal mask still applies** to every position in that combined
  sequence — a source token sitting at position 5 would be invisible to a decoder
  token at position 2, even though the whole point of the encoder is that the
  decoder should see _all_ source positions regardless of its own generation step.
- **Source length and target length don't move together.** Our reversal task fixes
  $S_{\text{src}} = 4$, but a real translation task has $S_{\text{src}} \ne T_{\text{tgt}}$
  in general, and $T$ grows by one every generation step while $S$ never changes. A
  single shared self-attention block would need the combined sequence length to be
  re-declared every step — there is no fixed $(S{+}T) \times (S{+}T)$ matrix to mask.

Cross-attention solves both at once by keeping $Q$ and $K/V$ as two genuinely
separate streams, computed once (the encoder side) and re-queried at every step
(the decoder side):

$$Q = \text{decoder state} \cdot W_Q \qquad K = \text{encoder output} \cdot W_K \qquad V = \text{encoder output} \cdot W_V$$

The decoder asks: _"given what I have generated so far ($Q$), which part of the
source text ($K, V$) do I need next?"_

### Correcting a common shorthand: "decoder attention is always causal"

A common mental shortcut is that every attention computation inside a decoder block
must be causally masked, because "decoders generate autoregressively." That shortcut
is only true for the decoder's **self**-attention sub-layer. Cross-attention has
**no mask at all** on the encoder side — not a relaxed mask, none whatsoever —
because $Q$ and $K/V$ come from two different sequences that were never generated
in a shared order in the first place. There is no "future" to hide from a query
that isn't itself part of the sequence being masked.

Because the encoder output is computed once and held fixed, the decoder re-queries
the full source map at every generation step with zero re-computation cost.

Notice the **asymmetry**: the score matrix is $(T_{\text{tgt}} \times S_{\text{src}})$,
not the $(S \times S)$ of self-attention. No mask is applied to the encoder dimension —
the decoder is free to attend to **any** source position regardless of generation step.

```
Decoder self-attn   Q, K, V from decoder state   (causal mask applied)
         |
Cross-attention     Q from decoder, K/V from encoder   (no mask on encoder side)
         |
FFN                 per-token nonlinear transformation
```


### Cross-Attention: Decoder Queries, Encoder Keys and Values

![Cross-attention bridge where decoder hidden states create queries and encoder outputs create keys and values](images/cross-attention-bridge.png)

Cross-attention forms queries from decoder hidden states and keys and values from encoder outputs. Each target position receives a weighted mixture of source information, and the target length and source length may differ.


> **PyTorch → Keras:** the class takes two separate inputs — `decoder_x` (projected by `W_Q`) and `encoder_kv` (projected by `W_K`/`W_V`) — and `forward()` manually computes the asymmetric `(T, S)` score matrix via `torch.matmul` + `F.softmax`. **Keras/TF equivalent:** `tf.keras.layers.MultiHeadAttention(num_heads, key_dim)(query=decoder_x, value=encoder_kv, key=encoder_kv)` implements exactly this "query from one sequence, key/value from another" cross-attention pattern as a single built-in call — no manual `W_Q`/`W_K`/`W_V` `Dense` layers required.

In [ ]:
#  CrossAttention module: Q from decoder, K/V from encoder
#
# The decoder's Q vector asks: "what do I need from the source?"
# The encoder's K/V matrices answer: "here is what each source position contains."
# No mask is applied on the encoder (src) dimension.


class CrossAttention(nn.Module):
    """
    Cross-attention layer used inside each decoder block.

    forward(decoder_x, encoder_kv) ->  output (B, T, d_model),
                                        attn_weights (B, n_heads, T, S)
    """

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # projects decoder
        self.W_K = nn.Linear(d_model, d_model, bias=False)  # projects encoder
        self.W_V = nn.Linear(d_model, d_model, bias=False)  # projects encoder
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, decoder_x, encoder_kv):
        B, T, _ = decoder_x.shape
        S = encoder_kv.shape[1]

        Q = self.W_Q(decoder_x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(encoder_kv).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(encoder_kv).view(B, S, self.n_heads, self.d_k).transpose(1, 2)

        # Score matrix: (B, n_heads, T, S)  — asymmetric T x S
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_w = F.softmax(scores, dim=-1)  # (B, n_heads, T, S)

        out = torch.matmul(attn_w, V)  # (B, n_heads, T, d_k)

        # Merge heads back into a single (B, T, d_model) tensor
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_O(out), attn_w

With the class defined, run a small demo: one decoder query step attending over the
4 encoder positions for our running example, so the shapes and the attention
distribution are visible before `CrossAttention` gets wired into a full decoder block.


> **PyTorch → Keras:** `ca_demo(dec_dummy, enc_dummy)` calls the custom `CrossAttention.forward` positionally (decoder query first, encoder key/value second); `.detach().numpy()` detaches the returned attention weights from autograd before printing. **Keras/TF equivalent:** the built-in `MultiHeadAttention` layer instead uses named `query=`/`value=`/`key=` keyword arguments (order-independent), and `return_attention_scores=True` returns the weights directly as an eager tensor — `.numpy()` alone is enough to inspect them.

In [ ]:
#  Demo: 1 decoder query attending to 4 encoder positions
torch.manual_seed(42)

# Instantiate the cross-attention layer defined above
ca_demo = CrossAttention(D_MODEL, N_HEADS)
enc_dummy = torch.randn(1, SEQ_LEN, D_MODEL)  # encoder output for 4 src tokens
dec_dummy = torch.randn(1, 1, D_MODEL)  # one decoder step

# Run one decoder query against all four encoder positions
ca_out, ca_w = ca_demo(dec_dummy, enc_dummy)

print("Cross-attention shapes:")
print(f"  Decoder Q   : {tuple(dec_dummy.shape)}  (1 decoder token)")
print(f"  Encoder K/V : {tuple(enc_dummy.shape)}  ({SEQ_LEN} source tokens)")
print(f"  Score matrix: {tuple(ca_w.shape)}")
print(f"  Output      : {tuple(ca_out.shape)}")
print()

# Show where attention mass lands (head 0, query 0)
w_h0 = ca_w[0, 0, 0].detach().numpy()
print("Attention weights over source positions (head 0, 1 query step):")
for pos, w in enumerate(w_h0):
    bar = "#" * int(w * 30)
    print(f"  src[{pos}]: {w:.3f}  {bar}")
print()
print("  -> The single decoder query scored ALL source positions.")
print("  -> Which source position gets the highest weight is learned from the")
print("     downstream prediction loss, not hard-coded.")

### Code Walkthrough: CrossAttention Module, Recapped

**The class and its demo above, split into two cells — 3 key patterns worth re-emphasizing:**

---

**`W_Q` projects decoder; `W_K` / `W_V` project encoder — two input streams**
The three projection matrices draw from different sources: `W_Q(decoder_x)` converts the decoder's current state into a query ("what do I need right now?"), while `W_K(encoder_kv)` and `W_V(encoder_kv)` convert all encoder positions into keys and values ("what does each source position contain?"). This is the exact architectural moment where encoder and decoder communicate.

---

**Score matrix `(B, n_heads, T, S)` — asymmetric by design**
`torch.matmul(Q, K.transpose(-2, -1))` produces a `(T, S)` matrix per head — not the `(S, S)` square of self-attention. `T` is the current decoder sequence length; `S` is the fixed source length. As the decoder generates more tokens, T grows; S never changes because the encoder output is computed once and held frozen.

```python
scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
# Q: (B, n_heads, T, d_k)    K^T: (B, n_heads, d_k, S)
# scores: (B, n_heads, T, S)  ← T x S, NOT S x S
```

---

**Demo bar chart — attention weights before training are roughly uniform**
The printed bar chart shows one decoder step distributing probability mass across all 4 source positions. Before training this is near-uniform (random projection). After 30 epochs of reversal training, step 0 should concentrate mass on source position 3 (the last source token), step 1 on position 2, etc. — the anti-diagonal pattern confirmed by the heatmap further down this notebook (Part 6).


#### Your turn — does cross-attention stay well-defined when T ≠ S?

The demo above ran exactly 1 decoder step against `SEQ_LEN=4` source positions.
Part 4 noted the cross-attention score matrix is asymmetric:
$(T_{\text{tgt}} \times S_{\text{src}})$, not $(S \times S)$.

**Predict:** if the decoder has **6** steps but the source still has 4 positions,
what shape will the cross-attention score matrix be — `(6, 4)`, `(4, 6)`, or `(6, 6)`?

Change `ex_t_steps` below and re-run to check.


> **PyTorch → Keras:** varying `ex_t_steps` while keeping `SEQ_LEN` fixed demonstrates that `CrossAttention.forward` accepts differently-shaped `decoder_x`/`encoder_kv` tensors at call time — PyTorch modules have no fixed input shape until you actually call `forward()`. **Keras/TF equivalent:** a Keras layer built with `build(input_shape)` on its first call can behave the same way for varying sequence length, but if the layer was built once with a static `input_shape` (e.g. inside a `Sequential` model with an explicit `Input` shape), passing a different `T` on a later call would raise a shape-mismatch error rather than silently working.

In [ ]:
#  EXERCISE — cross-attention shape when T != S
# Change ex_t_steps (decoder length) independent of the source length (SEQ_LEN=4)
# and predict the resulting score-matrix shape BEFORE running.

ex_t_steps = 6  # CHANGE: try 1, 4, 6, 10 — source length stays fixed at SEQ_LEN

torch.manual_seed(42)

# Fresh cross-attention layer for this shape experiment
ca_ex = CrossAttention(D_MODEL, N_HEADS)
enc_ex = torch.randn(1, SEQ_LEN, D_MODEL)  # source: SEQ_LEN positions, fixed
dec_ex = torch.randn(1, ex_t_steps, D_MODEL)  # target: ex_t_steps positions, varies

# Run cross-attention with mismatched T and S to inspect the resulting shapes
out_ex, w_ex = ca_ex(dec_ex, enc_ex)

print(f"Decoder steps (T) = {ex_t_steps},  Source positions (S) = {SEQ_LEN}")
print(
    f"  Score matrix shape : {tuple(w_ex.shape)}  -> (batch, n_heads, T={ex_t_steps}, S={SEQ_LEN})"
)
print(
    f"  Output shape       : {tuple(out_ex.shape)}  -> (batch, T={ex_t_steps}, d_model)"
)
print()

# Confirm the score matrix shape matches the predicted (T, S) asymmetry
if w_ex.shape[-2] == ex_t_steps and w_ex.shape[-1] == SEQ_LEN:
    print(
        f"  -> Confirmed: cross-attention is asymmetric ({ex_t_steps} x {SEQ_LEN}), never (S x S)."
    )
    print(
        "  -> The decoder can take ANY number of steps; the source length never changes shape."
    )

#### Predict before you run — what will the trained cross-attention map look like?

After training on the reversal task, the cross-attention map should show a
systematic pattern.

For the sequence `[3, 1, 4, 1]` -> `[1, 4, 1, 3]`:

- Decoder step 0 must output `1` — which is at **source position 3**
- Decoder step 1 must output `4` — which is at **source position 2**
- Decoder step 2 must output `1` — which is at **source position 1**
- Decoder step 3 must output `3` — which is at **source position 0**

**Predict:** the trained cross-attention map will look like:

A. The identity matrix (high weight on the diagonal: step 0 attends to src 0, etc.)
B. The anti-diagonal (step 0 attends to src 3, step 1 to src 2, etc.)
C. Uniform attention (each step spreads weight equally over all source positions)

Write your answer. Part 6 reveals it.


---

## Part 5 — Full Encoder-Decoder: Training

### Wiring encoder + cross-attention + decoder

The complete model stacks three components:

1. **Encoder** — bidirectional blocks; produces source map $(B, S, D)$
2. **Decoder** — causal self-attention + cross-attention at every block; generates
   target logits $(B, T, \text{vocab})$
3. **Language model head** — linear projection from $D$ to vocabulary size

The decoder input during training is the **teacher-forced** target:
`[BOS] + target[:-1]`. The decoder output is shifted: `target + [EOS]`.
The loss is cross-entropy averaged over all target positions.


> **PyTorch → Keras:** `DecoderBlock.forward` chains three residual sublayers in a fixed order — causal self-attention, then cross-attention against `encoder_out`, then the feed-forward network — each wrapped by its own `nn.LayerNorm` explicitly called inline. **Keras/TF equivalent:** this mirrors KerasNLP's `TransformerDecoder` building block, which bundles the same three sublayers behind one `call(decoder_sequence, encoder_sequence, ...)`; here each sublayer and its residual/normalisation wiring is written out by hand instead of provided by a single layer class.

In [ ]:
#  DecoderBlock (causal self-attn + cross-attn + FFN)
#
# Three sub-layers following Vaswani et al. (2017):
#   1. Causal self-attention  — decoder reads its own past generated tokens
#   2. Cross-attention        — decoder queries the encoder source map
#   3. Feed-forward           — per-position nonlinear transformation
# Each sub-layer uses a pre-norm residual connection.


class DecoderBlock(nn.Module):
    """Decoder layer: causal self-attn + cross-attn + FFN."""

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = CrossAttention(d_model, n_heads)
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, encoder_out, causal_mask=None):

        # Sub-layer 1: causal self-attention over previously generated decoder tokens
        sa_out, sa_w = self.self_attn(self.norm1(x), mask=causal_mask)
        x = x + sa_out

        # Sub-layer 2: cross-attention over the encoder's output
        ca_out, ca_w = self.cross_attn(self.norm2(x), encoder_out)
        x = x + ca_out

        # Sub-layer 3: position-wise feed-forward
        x = x + self.ffn(self.norm3(x))
        return x, sa_w, ca_w

With `MiniEncoder` and `DecoderBlock` both defined, wire them into one `EncoderDecoder` module:
encode the source once, then run the `DecoderBlock` stack over the teacher-forced target,
projecting the final hidden state to vocabulary logits with `lm_head`.


> **PyTorch → Keras:** the full model is a single `nn.Module` exposing three methods — `encode()`, `decode()`, and `forward()` — that a caller can invoke separately (e.g. `model.encode(src_ids)` once, then `model.decode(...)` repeatedly), because PyTorch lets you call any method on a module, not just `forward`. **Keras/TF equivalent:** a subclassed `tf.keras.Model` only auto-dispatches through `call()` when you invoke the model instance directly (`model(x)`); calling a custom `encode`/`decode` method works too, but you lose the automatic `training=`/mask-propagation plumbing that `call()` gets from the base `Model` class — those would need to be threaded through manually.

In [ ]:
#  EncoderDecoder: wires MiniEncoder + a DecoderBlock stack + an LM head
#
# encode(src_ids)   runs MiniEncoder ONCE per source sequence.
# decode(...)       runs the DecoderBlock stack over the (teacher-forced) target,
#                   re-using the same encoder_out at every generation step.
# forward(...)      builds the causal mask, then calls encode() then decode() —
#                   this is what every training/eval cell below actually calls.


class EncoderDecoder(nn.Module):
    """
    Full encoder-decoder transformer: a bidirectional MiniEncoder + a stack of
    DecoderBlocks (causal self-attn + cross-attn + FFN) + a linear LM head.
    forward(src_ids, tgt_ids) -> logits (B, T, vocab_size), ca_w (B, n_heads, T, S)
    ca_w is the cross-attention weight from the LAST decoder block only.
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.encoder = MiniEncoder(
            vocab_size, d_model, n_heads, d_ff, n_layers, max_seq
        )
        self.dec_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.register_buffer("dec_pe", sinusoidal_pe(max_seq, d_model))

        # Stack n_layers decoder blocks, each with self-attn + cross-attn + FFN
        self.dec_blocks = nn.ModuleList(
            [DecoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )
        self.dec_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def encode(self, src_ids):
        """Run the bidirectional encoder ONCE; its output is reused by every decoder step."""
        encoder_out, _ = self.encoder(src_ids)
        return encoder_out

    def decode(self, tgt_ids, encoder_out, causal_mask):
        """Run the causal self-attn + cross-attn + FFN stack over tgt_ids."""
        T = tgt_ids.shape[1]

        # Add positional info to target-token embeddings, trimmed to T steps
        x = self.dec_emb(tgt_ids) + self.dec_pe[:T]
        ca_w = None

        # Run every decoder block; keep only the last block's cross-attention weights
        for block in self.dec_blocks:
            x, _, ca_w = block(x, encoder_out, causal_mask=causal_mask)
        x = self.dec_norm(x)
        return self.lm_head(x), ca_w

    def forward(self, src_ids, tgt_ids):

        # Encode the source once before generating anything
        encoder_out = self.encode(src_ids)
        T = tgt_ids.shape[1]

        # diagonal=1 leaves (i, i) unmasked; only strictly-future positions (j > i) are blocked
        causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        return self.decode(tgt_ids, encoder_out, causal_mask)

Instantiate the full `EncoderDecoder`, count its parameters, and run one forward
pass on our running example to confirm every shape lines up before training it.


> **PyTorch → Keras:** `sum(p.numel() for p in model.parameters())` manually sums the element count of every registered parameter tensor to get the total parameter count. **Keras/TF equivalent:** `model.count_params()` (or `model.summary()` for a full per-layer breakdown) gives the same total in one call — Keras models track parameter counts natively without iterating `model.parameters()` yourself.

In [ ]:
#  Architecture inspection
torch.manual_seed(42)
model = EncoderDecoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
n_params = sum(p.numel() for p in model.parameters())
print(
    f"EncoderDecoder: vocab={VOCAB_SIZE}, d_model={D_MODEL}, "
    f"n_heads={N_HEADS}, d_ff={D_FF}, n_layers={N_LAYERS}"
)
print(f"Total parameters: {n_params:,}")
print()
src_t = torch.tensor([[3, 1, 4, 1]])
tgt_t = torch.tensor([[BOS, 1, 4, 1]])
logits, ca_w = model(src_t, tgt_t)
print(f"Forward pass: src {tuple(src_t.shape)}  tgt_in {tuple(tgt_t.shape)}")
print(f"  -> logits {tuple(logits.shape)}   (B, T, vocab_size)")
print(f"  -> ca_w   {tuple(ca_w.shape)}  (B, n_heads, T, S)")

### Code Walkthrough: Full Encoder-Decoder Architecture

**What just ran — 4 key patterns:**

---

**`DecoderBlock.forward` — three sub-layers in strict order**

The three operations must run in this sequence: (1) causal self-attention on the decoder's own past, (2) cross-attention querying the encoder, (3) feed-forward. The ordering matters: self-attention first lets the decoder incorporate its own generated context before querying the encoder, so the cross-attention Q already "knows what has been generated so far."

---

**`EncoderDecoder.encode(src_ids)` — reuses `MiniEncoder`, run once, reused by every decoder block**

`self.encoder(src_ids)` calls the already-defined, already-sanity-checked `MiniEncoder` and returns `(B, S, d_model)` — one enriched vector per source token. This tensor is passed as `encoder_out` to every `DecoderBlock` inside `decode()`. Because the encoder runs only **once** per source sequence, generating 100 output tokens costs just 1 encoder pass plus 100 decoder passes. Re-encoding at every step would be 100× more expensive.

---

**`causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)` — future tokens blocked**

`diagonal=1` leaves `(i, i)` unmasked — each position can attend to itself. Only positions `j > i` (future tokens) are masked with `-inf` before softmax. Without this mask, the decoder could trivially "cheat" during training by reading the answer token directly from the shifted target sequence.

---

**`self.lm_head = nn.Linear(d_model, vocab_size, bias=False)` — projects hidden state to vocabulary**

The final linear layer maps each decoder position's `d_model`-dimensional vector to `vocab_size` unnormalised logits. Cross-entropy loss is computed against target token IDs. The `bias=False` convention (standard in large transformers) is consistent with weight tying between the input embedding and the output projection — this notebook does not tie those weights (see the closing ledger), but keeps the convention for consistency with production code.

> **PyTorch shape note:** `logits` is `(B, T, vocab_size)`. `.view(-1, vocab_size)` flattens batch and time dims for `nn.CrossEntropyLoss`, giving `(B*T, vocab_size)` predictions against `(B*T,)` targets — loss is averaged over all generated positions.


> **PyTorch → Keras:** this cell hand-writes the entire training loop — `data_utils.DataLoader` batches a `TensorDataset`; each step calls `optimizer.zero_grad()`, computes `loss.backward()` to populate `.grad` on every parameter, clips gradients with `clip_grad_norm_`, then `optimizer.step()` applies the update; `model.train()`/`model.eval()` toggle dropout/batchnorm behaviour explicitly. **Keras/TF equivalent:** `model.compile(optimizer=Adam(3e-3), loss=SparseCategoricalCrossentropy(...))` followed by `model.fit(train_dataset, validation_data=val_dataset, epochs=30)` replaces this entire loop — gradient computation, clipping (`clipnorm=1.0` on the optimizer), the backward pass, and train/eval-mode switching are all handled internally by `fit()`.

In [ ]:
#  Training loop: teacher-forced seq2seq on reversal task
#
# Teacher forcing:
#   decoder input  = [BOS] + target[:-1]
#   decoder target = target + [EOS]
# Loss: cross-entropy over all target positions (including EOS).

import torch.utils.data as data_utils


def build_dataset(srcs, tgts):
    """Pack lists of int sequences into TensorDataset."""
    src_t = torch.tensor(srcs, dtype=torch.long)
    tgt_in = torch.cat(
        [
            torch.full((len(tgts), 1), BOS, dtype=torch.long),
            torch.tensor(tgts, dtype=torch.long),
        ],
        dim=1,
    )
    tgt_out = torch.cat(
        [
            torch.tensor(tgts, dtype=torch.long),
            torch.full((len(tgts), 1), EOS, dtype=torch.long),
        ],
        dim=1,
    )
    return data_utils.TensorDataset(src_t, tgt_in, tgt_out)


train_ds = build_dataset(train_srcs, train_tgts)
val_ds = build_dataset(val_srcs, val_tgts)
train_loader = data_utils.DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = data_utils.DataLoader(val_ds, batch_size=64, shuffle=False)

torch.manual_seed(42)
model = EncoderDecoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

EPOCHS = 30
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss = 0.0
    for src, tgt_in, tgt_out in train_loader:
        optimizer.zero_grad()
        logits, _ = model(src, tgt_in)
        loss = criterion(logits.view(-1, VOCAB_SIZE), tgt_out.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    ep_loss /= len(train_loader)
    train_losses.append(ep_loss)

    model.eval()
    with torch.no_grad():
        v_loss = sum(
            criterion(model(s, ti)[0].view(-1, VOCAB_SIZE), to.view(-1)).item()
            for s, ti, to in val_loader
        ) / len(val_loader)
    val_losses.append(v_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  train={ep_loss:.4f}  val={v_loss:.4f}")

print()
print(f"Final train loss : {train_losses[-1]:.4f}")
print(f"Final val   loss : {val_losses[-1]:.4f}")
print("  -> Random baseline loss: ~2.56 (log(13), uniform over 13 tokens)")

### Teacher-Forced Inner Mechanics: Source Once, Target Positions Together

Encoder-decoder training adds one path to the decoder-only update cycle: the source is encoded once, and every target position can cross-attend to that source representation. The target side remains causal.

Teacher forcing creates two aligned target sequences:

- decoder input: `[BOS] + target`
- expected output: `target + [EOS]`

A translation makes the alignment concrete. Suppose the training pair is:

```text
source: Working at a bank
target: Travailler dans une banque
```

At the word level, for intuition, the shifted target looks like this:

| Target position | Decoder receives | It must predict | Source access |
| --- | --- | --- | --- |
| 0 | `<BOS>` | `Travailler` | all encoded source positions |
| 1 | `<BOS> Travailler` | `dans` | all encoded source positions |
| 2 | `<BOS> Travailler dans` | `une` | all encoded source positions |
| 3 | `<BOS> Travailler dans une` | `banque` | all encoded source positions |
| 4 | `... une banque` | `<EOS>` | all encoded source positions |

These rows describe different causal views, but training does **not** run five decoder calls. The complete shifted target enters one masked decoder call, so all five logits and losses are computed in parallel. The losses combine into one scalar; its gradients flow backward through the LM head, decoder self-attention, cross-attention, and encoder. Unless parameters are deliberately frozen, both sides learn from the translation error.

At free-running inference, the schedule changes:

```text
encode `Working at a bank` once
<BOS>                           -> Travailler
<BOS> Travailler                -> dans
<BOS> Travailler dans           -> une
<BOS> Travailler dans une       -> banque
<BOS> Travailler dans une banque -> <EOS> (stop)
```

Now each decoder call must wait for the token chosen by the previous call. Cross-attention can revisit the full encoded source at every step, but causal self-attention can use only the target prefix generated so far. This is why training can supervise target positions together while inference must reveal them sequentially.

> **Tokenization note:** the table deliberately uses words to expose the sequence logic. T5 and BART operate on subword tokens, so one displayed word may occupy several target positions.

```mermaid
flowchart TD
    S["Source tokens"] --> E["Encode source once"]
    I["Shifted target input<br/>BOS + target"] --> D["Causal decoder positions<br/>computed in parallel"]
    E --> D
    D --> L["Vocabulary logits<br/>one vector per target position"]
    Y["Expected target<br/>target + EOS"] --> P["Per-position losses"]
    L --> P
    P --> A["Mean batch loss"]
    A --> B["One backward pass"]
    B --> U["One optimizer step"]
```

The next cell traces one isolated update without changing the trained reversal model used later.

In [ ]:
# Trace one teacher-forced update on an isolated model copy.
teacher_src, teacher_in, teacher_out = (
    tensor.unsqueeze(0) for tensor in train_ds[0]
)
teacher_probe = EncoderDecoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
teacher_probe.load_state_dict(model.state_dict())
teacher_optimizer = torch.optim.SGD(teacher_probe.parameters(), lr=1e-2)


def token_name(token_id):
    if token_id < 10:
        return str(token_id)
    return {PAD: "<PAD>", BOS: "<BOS>", EOS: "<EOS>"}[token_id]


def teacher_forced_losses(probe_model):
    logits, cross_attention = probe_model(teacher_src, teacher_in)
    losses = F.cross_entropy(
        logits.transpose(1, 2),
        teacher_out,
        reduction="none",
        ignore_index=PAD,
    )[0]
    return losses, cross_attention


teacher_optimizer.zero_grad()
before_losses, cross_attention = teacher_forced_losses(teacher_probe)
before_mean = before_losses.mean()
before_head = teacher_probe.lm_head.weight.detach().clone()
before_mean.backward()
gradient_norm = torch.sqrt(
    sum(
        parameter.grad.detach().pow(2).sum()
        for parameter in teacher_probe.parameters()
        if parameter.grad is not None
    )
)
teacher_optimizer.step()

with torch.no_grad():
    after_losses, _ = teacher_forced_losses(teacher_probe)
head_delta = (teacher_probe.lm_head.weight - before_head).norm()

print("Decoder input -> expected token -> loss contribution")
for position, token_loss in enumerate(before_losses):
    seen = token_name(teacher_in[0, position].item())
    expected = token_name(teacher_out[0, position].item())
    print(f"{seen:8s} -> {expected:8s} -> {token_loss.item():.4f}")

print(f"Cross-attention shape: {tuple(cross_attention.shape)} = (batch, heads, target, source)")
print(f"Mean loss before:      {before_mean.item():.4f}")
print(f"Global gradient norm:  {gradient_norm.item():.4f}")
print(f"LM-head update norm:   {head_delta.item():.6f}")
print(f"Mean loss after:       {after_losses.mean().item():.4f}")

assert torch.isfinite(gradient_norm)
assert head_delta > 0
print("PASS: all teacher-forced target losses produced one model update.")

### Code Walkthrough: Teacher-Forced Seq2Seq Training Loop

**What just ran — 4 key patterns:**

---

**`build_dataset` — teacher forcing: decoder input is the target shifted right**
The decoder receives `[BOS] + target[:-1]` as input and is asked to predict `target + [EOS]`. At training step `t`, the decoder sees the _ground-truth_ token `t−1`, not its own previous prediction. This "teacher forcing" makes gradients clean and convergence fast — but creates a training/inference mismatch: at inference the decoder must use its own (possibly wrong) outputs.

---

**`nn.CrossEntropyLoss(ignore_index=PAD)` — padding positions contribute zero gradient**
Sequences in a batch are padded to the same length. Without `ignore_index=PAD`, the model would waste gradient steps learning to predict padding tokens (which carry no meaning). With it, padded positions are silently excluded from the loss — only real token positions drive learning.

---

**`clip_grad_norm_(model.parameters(), 1.0)` — prevents exploding gradients**
Transformer training occasionally produces very large gradient norms when a loss spike occurs. Clipping the entire gradient vector to unit norm ensures no single noisy batch can undo hundreds of stable update steps. The threshold `1.0` is the standard default for transformer-scale training.

---

**`model.eval()` + `torch.no_grad()` in the validation block — two distinct switches**
`model.eval()` disables dropout and switches batch-norm to running statistics. `torch.no_grad()` stops PyTorch building a computation graph, saving ~40% of memory. Omitting either causes silent bugs: `model.train()` mode gives valid loss numbers but wastes memory and can change behaviour if dropout layers are present.


#### Predict before you run — what accuracy will the trained model achieve?

We just trained for 30 epochs on 2,000 reversal examples with `d_model=32`.

**Predict:** The validation sequence accuracy (all 4 digits must be correct to count) will be:

A. Below 50% — the model barely learns
B. 50-85% — partial learning, many errors
C. Above 90% — the model has cracked the reversal pattern

Write your answer, then run the loss-curve-and-accuracy cell below to measure it.


> **PyTorch → Keras:** `model.eval()` + `torch.no_grad()` wrap the evaluation forward passes, and `logits.argmax(dim=-1)` picks the highest-probability token id per position from the raw logit tensor. **Keras/TF equivalent:** `model.predict(...)` (or a plain call with `training=False`) is the evaluation-mode equivalent, and `tf.argmax(logits, axis=-1)` is the direct analogue of `.argmax(dim=-1)` — note the axis keyword name differs (`axis=` vs. `dim=`).

In [ ]:
#  Training loss curve + validation sequence accuracy

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, "b-o", ms=3, label="Train loss")
ax.plot(range(1, EPOCHS + 1), val_losses, "r-o", ms=3, label="Val   loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("EncoderDecoder training curve — sequence reversal task")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Sequence-level accuracy: all SEQ_LEN tokens must be correct
model.eval()
correct = total = 0
with torch.no_grad():
    for src, tgt_in, tgt_out in val_loader:
        logits, _ = model(src, tgt_in)
        preds = logits.argmax(dim=-1)  # (B, T)
        match = (preds[:, :SEQ_LEN] == tgt_out[:, :SEQ_LEN]).all(dim=1)
        correct += match.sum().item()
        total += src.shape[0]

accuracy = correct / total
print(
    f"Validation sequence accuracy: {accuracy:.1%}  ({correct}/{total} fully correct)"
)
print()
if accuracy > 0.90:
    print("  -> Excellent: the model has learned the reversal pattern.")
elif accuracy > 0.70:
    print("  -> Good: most sequences correct; a few more epochs would help.")
else:
    print("  -> Still converging — try more epochs or a larger model.")
print()
print("  -> Random baseline: (1/10)^4 = 0.01% (guessing each digit independently)")

#### What just happened — and what is missing

The model trained, loss fell, and validation accuracy is high.
But we have not verified _how_ it solved the task. Did it actually route
decoder step $i$ to source position $S-1-i$ through cross-attention?

Part 6 checks whether its cross-attention routing matches the reversal rule; Part 6a then tests genuine free-running generation without the answer key.


---

## Part 6 — The Cross-Attention Map

### Diagnosing the decoder's learned attention routing

For perfect reversal of `[3, 1, 4, 1]` to `[1, 4, 1, 3]`:

| Decoder step | Must output | Source token to attend to | Source position |
| ------------ | ----------- | ------------------------- | --------------- |
| 0            | 1           | 1 (last)                  | 3               |
| 1            | 4           | 4                         | 2               |
| 2            | 1           | 1                         | 1               |
| 3            | 3           | 3 (first)                 | 0               |

If the model learned this, the cross-attention map should be the **anti-diagonal**.
That would be evidence that the learned routing matches the reversal rule. The held-out sequence accuracy remains the stronger check that the model generalised beyond memorising training examples.


> **PyTorch → Keras:** `model(ex_src, ex_tgt_in)` under `torch.no_grad()` returns both the logits and the last decoder block's cross-attention weights (`ca_w_ex`), which are then `.detach().numpy()`'d for plotting. **Keras/TF equivalent:** extracting intermediate attention weights from a Keras model typically requires either building the model with `return_attention_scores=True` on the `MultiHeadAttention` layer and returning it explicitly from `call()`, or constructing a secondary `tf.keras.Model` whose outputs point at an inner layer's output — Keras has no automatic "every layer returns its internals" mechanism analogous to just returning extra values from `forward()`.

In [ ]:
#  Cross-attention heatmap: does decoder step i attend to source position S-1-i?
#
# We extract cross-attention weights from the last decoder block for
# the single example [3, 1, 4, 1] -> [1, 4, 1, 3].

model.eval()
ex_src = torch.tensor([[3, 1, 4, 1]])
ex_tgt_in = torch.tensor([[BOS, 1, 4, 1]])  # teacher-forced

with torch.no_grad():
    logits_ex, ca_w_ex = model(ex_src, ex_tgt_in)

# ca_w_ex: (1, n_heads, T, S)  where T = SEQ_LEN+1 (BOS + 4 targets)
ca_avg = ca_w_ex[0].mean(dim=0).detach().numpy()  # (T, S) avg over heads
ca_display = ca_avg[:SEQ_LEN, :]  # rows 0..3 (generating steps)

src_labels = ["3", "1", "4", "1"]
tgt_labels = ["->1 (step 0)", "->4 (step 1)", "->1 (step 2)", "->3 (step 3)"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: averaged over all heads
ax = axes[0]
sns.heatmap(
    ca_display,
    ax=ax,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    xticklabels=src_labels,
    yticklabels=tgt_labels,
    linewidths=0.5,
    vmin=0,
    vmax=1,
)
ax.set_title(
    "Cross-attention (avg all heads)\nDecoder step vs. Source position", fontsize=10
)
ax.set_xlabel("Source position (key)")
ax.set_ylabel("Decoder step (query)")

# Right: head 0 only
ax2 = axes[1]
ca_h0 = ca_w_ex[0, 0, :SEQ_LEN, :].detach().numpy()
sns.heatmap(
    ca_h0,
    ax=ax2,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=src_labels,
    yticklabels=tgt_labels,
    linewidths=0.5,
    vmin=0,
    vmax=1,
)
ax2.set_title("Cross-attention (head 0 only)", fontsize=10)
ax2.set_xlabel("Source position (key)")
ax2.set_ylabel("Decoder step (query)")

plt.suptitle(
    f"Cross-attention map: [3,1,4,1] -> [1,4,1,3]", fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Quantify anti-diagonal alignment
anti_diag = sum(ca_display[i, SEQ_LEN - 1 - i] for i in range(SEQ_LEN)) / SEQ_LEN
print(f"Mean attention weight on anti-diagonal positions: {anti_diag:.3f}")
print()
if anti_diag > 0.5:
    print("  -> Strong anti-diagonal pattern confirmed!")
    print("     Decoder step i attends most to source position S-1-i.")
    print("     Answer to the Part 4 prediction: B (anti-diagonal).")
else:
    print("  -> Attention is more diffuse. Try training for more epochs.")
    print("     The pattern should emerge with sufficient convergence.")
print()
preds = logits_ex.argmax(dim=-1)[0, :SEQ_LEN].tolist()
gold = [1, 4, 1, 3]
print(f"Model prediction (greedy): {preds}")
print(f"Gold target               : {gold}")
correct_str = "Correct!" if preds == gold else "Incorrect — try more training epochs."
print(f"  -> {correct_str}")

In [ ]:
#  FuncAnimation: cross-attention anti-diagonal assembles step by step
# Each frame reveals one more decoder step row in the heatmap.
# Grey cells = steps not yet generated; coloured cells = steps already decided.
# Watch how the diagonal spotlight moves from bottom-right to top-left,
# proving the decoder queries a different source position at each generation step.

from matplotlib.animation import FuncAnimation
from matplotlib.patches import Patch
from IPython.display import HTML, display
import matplotlib as mpl

n_tgt = ca_display.shape[0]  # T = SEQ_LEN decoder steps
n_src = ca_display.shape[1]  # S = source positions

fig_anim, ax_anim = plt.subplots(figsize=(6, 4.5))

# Static legend handles for the categorical grey/coloured distinction below.
# ax_anim.clear() runs every frame, so the legend is re-added inside
# update_ca_anim rather than once outside it — the handles themselves never
# change, only their re-draw is repeated per frame (Section 10.2: an animated
# categorical color needs its own legend, not just a title string).
legend_handles = [
    Patch(facecolor="#b30000", edgecolor="black", label="Revealed step (generated)"),
    Patch(facecolor="#d3d3d3", edgecolor="black", label="Not yet generated (pending)"),
]


def update_ca_anim(frame):
    ax_anim.clear()

    # Reveal rows 0..frame; grey out the rest
    data = np.full_like(ca_display, np.nan)
    data[: frame + 1, :] = ca_display[: frame + 1, :]
    cmap_anim = mpl.colormaps["YlOrRd"].copy()
    cmap_anim.set_bad(color="#d3d3d3")

    # Only annotate on the final frame so numbers don't flicker
    annot = frame == n_tgt - 1
    y_labels = tgt_labels[: frame + 1] + ["(pending)"] * (n_tgt - frame - 1)
    sns.heatmap(
        data,
        ax=ax_anim,
        annot=annot,
        fmt=".2f",
        cmap=cmap_anim,
        xticklabels=src_labels,
        yticklabels=y_labels,
        linewidths=0.5,
        vmin=0,
        vmax=1,
        cbar=False,
    )
    ax_anim.set_title(
        f"Cross-attention: step {frame} of {n_tgt - 1} revealed\n"
        "(grey = token not yet generated)",
        fontsize=9,
    )
    ax_anim.set_xlabel("Source position (key)")
    ax_anim.set_ylabel("Decoder step (query)")
    ax_anim.tick_params(axis="y", labelsize=8)
    ax_anim.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.28),
        ncol=1,
        fontsize=7,
        frameon=True,
    )


anim = FuncAnimation(
    fig_anim,
    update_ca_anim,
    frames=n_tgt,
    interval=900,
    blit=False,
    repeat=True,
    repeat_delay=1800,
)
plt.close(fig_anim)

print("Watch the anti-diagonal attention pattern assemble itself:")
print("  Frame 0  (generate '1')  → attention peaks at src position 3  (last token)")
print("  Frame 1  (generate '4')  → attention shifts left to src position 2")
print("  Frame 2  (generate '1')  → attention at src position 1")
print("  Frame 3  (generate '3')  → attention at src position 0  (first token)")
print(
    "  The spotlight walks backward across the source — cross-attention learned reversal."
)
display(HTML(anim.to_jshtml(fps=1)))

#### What just happened — and what is missing

The heatmap is a useful diagnostic: cross-attention learned to route decoder step $i$ to
source position $S-1-i$, exactly the pattern required for reversal. The architecture
did not need to be told this — it emerged from the prediction loss.

But look closely at how that diagnostic was generated: `ex_tgt_in` above was the **ground-truth**
target, not the model's own predictions. Before bridging to real models, one more question needs
answering honestly: did the decoder actually _generate_ anything, or did every check so far quietly
lean on the answer key? The next subsection answers that directly.


---

### 6a. From Teacher-Forced Proof to Free-Running Generation

Every check so far — the validation accuracy in Part 5, the cross-attention heatmap above — fed the
**ground-truth** target into the decoder as `tgt_in` (teacher forcing). A common shorthand for what
we just did is _"the model generates the reversed sequence."_ Taken literally, that shorthand
implies the decoder produced `[1, 4, 1, 3]` token-by-token from its **own** previous guesses. That
is not what happened: at every one of the four steps above, the decoder was handed the correct
previous token from `tgt_in`, not its own prediction. If step 1 had guessed wrong, step 2 would
still have seen the correct token `1` as its input — errors cannot compound under teacher forcing.

Genuine autoregressive generation must feed each predicted token back in as the next step's input,
starting from nothing but `[BOS]`. This is also where a real training/inference mismatch — often
called **exposure bias** — lives: the decoder trains conditioned on gold history, but at real
inference time it only ever sees its own (possibly wrong) history.

**Predict before you run:** will free-running (self-fed) sequence accuracy on the validation set
be higher than, equal to, or lower than the teacher-forced accuracy measured in Part 5?


### Training Inputs and Free-Running Generation

![Teacher forcing during training compared with free-running autoregressive generation at inference](images/teacher-forcing-vs-generation.png)

During teacher forcing, the decoder receives the ground-truth previous target token while training against the next one. During inference, it feeds back its own generated tokens, which is why an early mistake can influence later steps.


> **PyTorch → Keras:** the loop calls `model.encode()` once, then repeatedly calls `model.decode(dec_in, encoder_out, causal_mask)` inside `torch.no_grad()`, growing `dec_in` one token at a time with `torch.cat([dec_in, next_token], dim=1)` — a fully manual autoregressive decode loop. **Keras/TF equivalent:** HuggingFace's TF models expose the same idea via `tf_model.generate(input_ids, max_new_tokens=...)`, which runs this exact encode-once/decode-repeatedly loop (plus KV-caching) internally; a hand-rolled Keras version would look almost identical to this cell, just with `tf.concat` instead of `torch.cat` and eager `.numpy()` calls instead of `.item()`/`.tolist()`.

In [ ]:
#  Greedy autoregressive decoding: feed each prediction back in as the next input
#
# Unlike every evaluation above, this loop NEVER sees the ground-truth target.
# It starts from [BOS] alone and grows the decoder input one predicted token at a time —
# real inference behaviour for T5/BART/any encoder-decoder model.

def greedy_decode(model, src_ids, max_len=SEQ_LEN):
    '''Autoregressive greedy decoding. Returns (B, max_len) predicted token ids.

    Parameters
    ----------
    model   : trained EncoderDecoder
    src_ids : (B, S) source token ids
    max_len : number of tokens to generate (reversal's output length always equals SEQ_LEN)
    '''
    model.eval()
    B = src_ids.shape[0]
    with torch.no_grad():
        encoder_out = model.encode(src_ids)
        dec_in = torch.full((B, 1), BOS, dtype=torch.long)   # start token only — no gold history
        for _ in range(max_len):
            T = dec_in.shape[1]
            causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
            logits, _ = model.decode(dec_in, encoder_out, causal_mask)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # greedy: highest-prob token
            dec_in = torch.cat([dec_in, next_token], dim=1)             # feed prediction back in
    return dec_in[:, 1:]   # drop the leading BOS


# Free-running accuracy over the full validation set (compare to the teacher-forced number above)
model.eval()
correct_free = total_free = 0
with torch.no_grad():
    for src, _, tgt_out in val_loader:
        preds_free = greedy_decode(model, src, max_len=SEQ_LEN)
        match = (preds_free == tgt_out[:, :SEQ_LEN]).all(dim=1)
        correct_free += match.sum().item()
        total_free   += src.shape[0]

free_accuracy = correct_free / total_free
print(f"Teacher-forced sequence accuracy (Part 5) : {accuracy:.1%}")
print(f"Free-running (greedy) sequence accuracy   : {free_accuracy:.1%}")
print()
if free_accuracy < accuracy:
    print("  -> Free-running accuracy is LOWER: one early wrong token compounds into every later")
    print("     step, because there is no gold history to fall back on. This is exposure bias,")
    print("     measured directly rather than just asserted.")
else:
    print("  -> Free-running accuracy matched (or exceeded) teacher-forced accuracy here — the")
    print("     model's own greedy predictions were reliable enough not to compound errors on")
    print("     this simple a task; exposure bias grows more visible on longer/harder sequences.")


#### Aside: how would beam search differ?

**This is NOT `model.generate(num_beams=k)`.** Greedy decoding commits to the single best token per
step and can't backtrack. Beam search keeps the top-_k_ partial sequences alive instead of one.


> **PyTorch → Keras:** `F.log_softmax(logits[0, -1, :], dim=-1)` converts final-step logits to log-probabilities, and `.topk(beam_width)` returns the top-`k` values and indices used to expand each beam. **Keras/TF equivalent:** `tf.nn.log_softmax(logits, axis=-1)` and `tf.math.top_k(logits, k=beam_width)` are the direct analogues; production beam search in TF/Keras is normally delegated to `model.generate(num_beams=k)` rather than hand-written, exactly as this cell's closing note points out for the PyTorch side too.

In [ ]:
#  Illustrative beam search: track top-k candidate sequences, not just 1
#
# Disclaimer: no length normalisation, no batching across beams, no early-stop on EOS —
# a from-scratch illustration of the core "keep top-k partial sequences" idea, not a
# production decoder. Reuses model.encode()/model.decode() from greedy_decode above.


def beam_search_decode(model, src_ids, beam_width=3, max_len=SEQ_LEN):
    """Illustrative beam search. Returns (best_tokens, best_log_prob) for the top-scoring sequence."""
    model.eval()
    assert src_ids.shape[0] == 1, "illustrative version: batch size 1 only"
    with torch.no_grad():
        encoder_out = model.encode(src_ids)

        # Each beam: (token_id_list, cumulative_log_prob)
        beams = [([BOS], 0.0)]
        for _ in range(max_len):
            candidates = []
            for tokens, log_prob in beams:
                dec_in = torch.tensor([tokens], dtype=torch.long)
                T = dec_in.shape[1]
                causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
                logits, _ = model.decode(dec_in, encoder_out, causal_mask)
                log_probs = F.log_softmax(logits[0, -1, :], dim=-1)
                topk_logp, topk_idx = log_probs.topk(beam_width)
                for lp, idx in zip(topk_logp.tolist(), topk_idx.tolist()):
                    candidates.append((tokens + [idx], log_prob + lp))

            # Keep only the top beam_width candidates across ALL beams' expansions
            candidates.sort(key=lambda c: c[1], reverse=True)
            beams = candidates[:beam_width]
    best_tokens, best_log_prob = beams[0]
    return best_tokens[1:], best_log_prob  # drop leading BOS


example_src = torch.tensor([[3, 1, 4, 1]])
greedy_tokens = greedy_decode(model, example_src, max_len=SEQ_LEN)[0].tolist()
beam_tokens, beam_logp = beam_search_decode(
    model, example_src, beam_width=3, max_len=SEQ_LEN
)

print(f"Example [3, 1, 4, 1]  (gold reversed = [1, 4, 1, 3])")
print(f"  Greedy decode      : {greedy_tokens}")
print(f"  Beam search (k=3)  : {beam_tokens}   (cumulative log-prob = {beam_logp:.3f})")
print()
outcome = (
    "matched greedy"
    if greedy_tokens == beam_tokens
    else "found a different, higher-scoring sequence"
)
print(
    f"  -> Beam search {outcome} — it pays off most when the single greedy path isn't globally best."
)
print()
print(
    "Production alternative: HuggingFace's model.generate(num_beams=3, ...) — adds length"
)
print(
    "normalisation, batched beam execution, and per-beam EOS handling on top of this idea."
)

#### What just happened — and what is missing

Free-running generation and beam search both reuse the identical encoder output and decoder
blocks — generation is an _inference-time_ choice, not a different model. What remains is bridging
this toy architecture to the production model families it mirrors. Part 7 does exactly that.


---

## Part 7 — Toy to Real: T5 / BART

### Parameter mapping table

Every hyperparameter in our toy model has a direct counterpart in production
encoder-decoder models. The architecture is identical; only the scale changes.

| Hyperparameter         | Toy (this notebook)  | T5-small                    | BART-base                     |
| ---------------------- | -------------------- | --------------------------- | ----------------------------- |
| `d_model`              | 32                   | 512                         | 768                           |
| `n_heads`              | 4                    | 8                           | 12                            |
| `d_ff`                 | 64                   | 2,048                       | 3,072                         |
| `n_layers` (each side) | 2                    | 6                           | 6                             |
| `vocab_size`           | 13                   | 32,128                      | 50,265                        |
| Parameters             | ~20 K                | ~60 M                       | ~139 M                        |
| Pre-training task      | Sequence reversal    | Span denoising (C4)         | Denoising (Books + Wikipedia) |
| Key use-case           | Toy proof-of-concept | Summarisation / translation | Summarisation / translation   |

Every class we wrote — `MultiHeadSelfAttention`, `CrossAttention`, `EncoderBlock`,
`DecoderBlock` — is a scaled-up copy of what lives inside T5 and BART.
The cross-attention formula $Q_{\text{dec}}(K_{\text{enc}})^\top / \sqrt{d_k}$ is
word-for-word identical; only the tensor widths differ.

**What "span denoising" and "denoising" actually mean (named above, not built here):** our toy
model is trained on a fully-supervised task — every source sequence has one known-correct target.
T5 and BART are _pre-trained_ instead on self-supervised denoising: a clean passage of real text is
corrupted (T5 masks out contiguous **spans** of tokens and asks the decoder to reconstruct just
those spans; BART additionally deletes, shuffles, or masks whole tokens/sentences and asks the
decoder to reconstruct the **entire** original passage), and the network learns general language
structure from millions of such examples before ever seeing a labelled translation/summarisation
pair. This notebook does not build either objective — reversal's supervised target is the simpler
setting to prove cross-attention itself; span-corruption/denoising is a data-construction technique
layered on top of the identical architecture built above.


### The Same Pattern at Production Scale

![Comparison of toy and production-scale encoder-decoder Transformers, including T5 and BART model families](images/toy-to-t5-bart.png)

Scaling adds depth, heads, representation width, context capacity, and training data while preserving the essential pattern: bidirectional source encoding, cross-attention, and causal target generation. T5 and BART are encoder-decoder model families with different pre-training objectives, not interchangeable implementations.

### What Is Reused During Generation?

`encode source once` is precise at the architectural boundary: the encoder produces hidden states $H_{enc}$ once for a source request, and those states do not change while the decoder generates its target. Inside each decoder layer, however, cross-attention applies that layer's own learned projections:

$$K^{(l)}_{enc}=H_{enc}W^{(l)}_K, quad V^{(l)}_{enc}=H_{enc}W^{(l)}_V$$

Because $H_{enc}$ is unchanged during inference, an implementation may precompute and cache these projected encoder keys and values for every decoder layer. That cache is not one universal `encoder memory`: each layer (and internally each head) has its own projected views. Decoder self-attention uses a separate cache that grows as each new target token is generated.

> **Fixed activations are not frozen parameters.** During inference, encoder activations are reused. During end-to-end training, target loss gradients still pass through cross-attention into the encoder unless the training setup explicitly freezes encoder parameters.


> **PyTorch → Keras:** `T5ForConditionalGeneration.from_pretrained("t5-small")` loads HuggingFace's PyTorch T5 implementation, and `t5_model.generate(**inputs, max_new_tokens=60)` runs beam/greedy decoding internally under the hood, wrapped here in `torch.no_grad()`. **Keras/TF equivalent:** HuggingFace also ships `TFT5ForConditionalGeneration.from_pretrained("t5-small")` with an identical `.generate(...)` API — no `torch.no_grad()` equivalent is needed since `generate()` already runs outside a gradient tape by default in TF.

In [ ]:
#  T5-small summarisation demo (HuggingFace Transformers)
#
# Guarded by try/except — works offline if t5-small weights are already cached.
# If not, a graceful message explains how to cache them.
# The toy model trained above uses the identical cross-attention mechanism.

try:
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    import warnings

    warnings.filterwarnings("ignore")

    print("Loading T5-small (may download ~240 MB on first run)...")
    tokenizer = T5Tokenizer.from_pretrained("t5-small", legacy=False)
    t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")
    t5_model.eval()

    text = (
        "summarize: The encoder-decoder transformer uses cross-attention to bridge "
        "a source sequence and a target sequence. The encoder reads the full source "
        "bidirectionally and produces a rich context map. The decoder generates output "
        "tokens autoregressively, querying the encoder context at every step. "
        "This architecture underlies T5, BART, and the original Transformer paper."
    )

    inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True)
    with torch.no_grad():
        out_ids = t5_model.generate(**inputs, max_new_tokens=60)
    summary = tokenizer.decode(out_ids[0], skip_special_tokens=True)

    print("T5-small summarisation:")
    print(f"  Input : {text[12:120]}...")
    print(f"  Output: {summary}")
    print()
    print("  -> T5 uses the exact same cross-attention: Q=decoder, K=V=encoder.")
    print("  -> Differences from our toy: d_model=512, n_heads=8, n_layers=6,")
    print("     vocab=32128, trained on C4 corpus (~750 GB text).")

except Exception as e:
    print(f"T5 demo skipped ({type(e).__name__}: {e})")
    print()
    print("To enable: pip install transformers  then re-run this cell.")
    print("  T5 weights (~240 MB) will be downloaded and cached automatically.")
    print()
    print(
        "The toy model you trained above uses the IDENTICAL cross-attention mechanism."
    )
    print("  Toy  d_model=32    T5-small d_model=512")
    print("  Toy  vocab=13      T5-small vocab=32,128")
    print("  Architecture: identical in every structural detail.")

#### Your turn — change one variable and predict

The model was trained on length-4 sequences with `d_model=32`.

```python
# In the Setup cell, change:
SEQ_LEN = 6    # CHANGE: what happens to the cross-attention map dimensions?
D_MODEL = 64   # CHANGE: more capacity — predict convergence speed?
```

**Predict before running:**

1. For `SEQ_LEN=6`, what is the shape of the cross-attention weight tensor?
2. Does the anti-diagonal pattern still appear for length-6 sequences?
3. Does doubling `D_MODEL` help or hurt training speed? (More capacity vs. more
   parameters to optimise.)

Then retrain and compare the cross-attention heatmap.


---

## Optional Appendix: Reader, Writer, Translator on the Original Running Example

The main chapter above built encoder-decoder mechanics at useful depth. This appendix preserves the original compact architecture-family comparison from the former all-in-one Transformer notebook. It returns to `the cat sat on the mat` and shows the same family distinction with a smaller model.

The executable recap cells intentionally re-establish the Part 1/Part 2 toy definitions in this kernel. The appendix then runs its original cells without replacing the canonical model you built above.

All appendix classes and globals use an `Appendix` prefix. Running this optional lab cannot replace the canonical classes defined in the main chapter.


### Appendix Bootstrap

These definitions were derived in Parts 1 and 2. They are repeated only so the preserved compact lab remains executable after the notebook split.


In [ ]:
#  Imports
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import seaborn as sns
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

warnings.filterwarnings('ignore')

# Plotly is optional -- fall back to matplotlib-only 3D plots if it isn't installed
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})
print(f"torch   {torch.__version__}")
print(f"numpy   {np.__version__}")


In [ ]:
#  Vocabulary
APPENDIX_VOCAB = {
    "<PAD>": 0, "<BOS>": 1, "<EOS>": 2,
    "the": 3, "a": 4, "cat": 5, "dog": 6,
    "mat": 7, "fence": 8, "sat": 9, "ran": 10,
    "jumped": 11, "on": 12, "over": 13, "big": 14,
}

# Invert the vocab so token IDs can be decoded back to words
APPENDIX_IDX2WORD = {v: k for k, v in APPENDIX_VOCAB.items()}
APPENDIX_VOCAB_SIZE = len(APPENDIX_VOCAB)

#  3D Semantic Embeddings (Concreteness, Animacy, Dynamism)
E = {
    "<PAD>": [0.00, 0.00, 0.00], "<BOS>": [0.08, 0.08, 0.15], "<EOS>": [0.08, 0.08, 0.15],
    "the":   [0.05, 0.04, 0.08], "a":     [0.05, 0.04, 0.08],
    "cat":   [0.91, 0.94, 0.38], "dog":   [0.88, 0.92, 0.55],
    "mat":   [0.96, 0.04, 0.04], "fence": [0.93, 0.03, 0.03],
    "sat":   [0.34, 0.18, 0.78], "ran":   [0.28, 0.12, 0.96],
    "jumped":[0.30, 0.14, 0.98], "on":    [0.14, 0.04, 0.18],
    "over":  [0.17, 0.04, 0.24], "big":   [0.44, 0.04, 0.09],
}

# Stack each token's embedding vector in vocab-ID order into one matrix
appendix_embedding_matrix = torch.tensor(
    [E[APPENDIX_IDX2WORD[i]] for i in range(APPENDIX_VOCAB_SIZE)], dtype=torch.float32
)

APPENDIX_SENTENCE = "the cat sat on the mat"
APPENDIX_TOKENS = APPENDIX_SENTENCE.split()

# Encode the running example sentence into vocab IDs
APPENDIX_TOKEN_IDS = [APPENDIX_VOCAB[w] for w in APPENDIX_TOKENS]
APPENDIX_SEQ_LEN = len(APPENDIX_TOKENS)

print(f"Vocab size       : {APPENDIX_VOCAB_SIZE}")
print(f"Embedding shape  : {tuple(appendix_embedding_matrix.shape)}  (vocab x 3D)")
print(f"Running sentence : {APPENDIX_SENTENCE!r}")
print(f"Token IDs        : {APPENDIX_TOKEN_IDS}")
print()
print("Embedding matrix -> Concreteness, Animacy, Dynamism:")

# Skip the special tokens (indices 0-2) when printing the embedding table
for word, vec in list(E.items())[3:]:
    print(f"  {word:<10} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}]")


In [ ]:
#  Tokeniser
# Convert a raw string into vocab IDs, optionally wrapping with BOS/EOS markers
def encode(text: str, add_bos: bool = False, add_eos: bool = False):
    ids = [APPENDIX_VOCAB.get(w, APPENDIX_VOCAB["<PAD>"]) for w in text.lower().split()]
    if add_bos:
        ids = [APPENDIX_VOCAB["<BOS>"]] + ids
    if add_eos:
        ids = ids + [APPENDIX_VOCAB["<EOS>"]]
    return ids


# Convert vocab IDs back into a whitespace-joined string
def decode(ids):
    return " ".join(APPENDIX_IDX2WORD.get(i, "<?>") for i in ids)


phrase = "the big cat jumped over the fence"
enc = encode(phrase)
dec = decode(enc)
print(f"Input  : {phrase!r}")
print(f"Encoded: {enc}")
print(f"Decoded: {dec!r}")
print()
print('With BOS/EOS markers:')
enc2 = encode(phrase, add_bos=True, add_eos=True)
print(f"  {enc2}")
print()
print("  -> One word = one token; BPE splits rare words in real models.")


In [ ]:
#  Sinusoidal Positional Encoding
def appendix_sinusoidal_pe(seq_len: int, d_model: int) -> torch.Tensor:
    """Classic additive positional encoding (Vaswani et al. 2017).
    Handles both even and odd d_model gracefully.
    """
    pe = np.zeros((seq_len, d_model), dtype=np.float32)
    positions = np.arange(seq_len)[:, None].astype(np.float32)
    dims = np.arange(0, d_model, 2).astype(np.float32)

    # Geometrically decaying frequency per dimension pair
    freqs = 1.0 / (10000 ** (dims / d_model))

    # Even dims get sine, odd dims get cosine, at the same frequency
    pe[:, 0::2] = np.sin(positions * freqs)
    n_cos = pe[:, 1::2].shape[1]
    pe[:, 1::2] = np.cos(positions * freqs[:n_cos])
    return torch.tensor(pe)


D_VIS = 16
pe_matrix = appendix_sinusoidal_pe(APPENDIX_SEQ_LEN, D_VIS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap: PE value at every (token position, dimension) pair
ax = axes[0]
im = ax.imshow(pe_matrix.numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(D_VIS))
ax.set_xticklabels([f'd{i}' for i in range(D_VIS)], fontsize=8, rotation=45)
ax.set_yticks(range(APPENDIX_SEQ_LEN)); ax.set_yticklabels(APPENDIX_TOKENS, fontsize=10)
ax.set_title('Sinusoidal PE - our sentence')
ax.set_xlabel('Embedding dimension'); ax.set_ylabel('Token position')
plt.colorbar(im, ax=ax)

# Line plot: a few dimensions over 50 positions, revealing fast vs. slow oscillation
ax2 = axes[1]
pe_long = appendix_sinusoidal_pe(50, D_VIS).numpy()
for i in [0, 2, 6, 14]:
    label = f'dim {i} - {"fast" if i < 4 else "slow"}'
    ax2.plot(pe_long[:, i], label=label, lw=1.8)
ax2.set_title('PE signal per dimension over 50 positions')
ax2.set_xlabel('Token position'); ax2.set_ylabel('PE value')
ax2.legend(fontsize=8); ax2.set_ylim(-1.1, 1.1)

plt.suptitle('Sinusoidal Positional Encoding', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Add the 3D positional signal directly onto the token embeddings
emb_vectors = appendix_embedding_matrix[APPENDIX_TOKEN_IDS]
pe_3d = appendix_sinusoidal_pe(APPENDIX_SEQ_LEN, 3)
enriched = emb_vectors + pe_3d
print('After adding 3D sinusoidal PE:')
for i, w in enumerate(APPENDIX_TOKENS):

    # Format a 3-vector for aligned printing
    def fmt(v_list):
        return f'[{v_list[0]:+.3f}, {v_list[1]:+.3f}, {v_list[2]:+.3f}]'
    print(f'[{i}] {w:<8}  orig={fmt(emb_vectors[i].tolist())}  pe={fmt(pe_3d[i].tolist())}  sum={fmt(enriched[i].tolist())}')


In [ ]:
#  Working model constants
APPENDIX_D_WORK = 16    # functional model dimension
APPENDIX_NUM_HEADS = 2  # attention heads
APPENDIX_D_HEAD = APPENDIX_D_WORK // APPENDIX_NUM_HEADS   # 8 per head
APPENDIX_D_FF = 32      # feed-forward hidden size


# Splits Q/K/V into n_heads parallel attention heads, then concatenates and projects back
class AppendixMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    # Reshape into (B, H, S, d_head), run scaled dot-product attention per head,
    # then merge the heads back together
    def forward(self, x, mask=None):
        B, S, _ = x.shape
        Q_mh = self.W_Q(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)  # (B, H, S, d_head)
        K_mh = self.W_K(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        V_mh = self.W_V(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = (Q_mh @ K_mh.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, H, S, S)
        if mask is not None:
            scores = scores.masked_fill(mask.bool(), float('-inf'))
        attn_w_mh = torch.softmax(scores, dim=-1)   # (B, H, S, S)
        out = attn_w_mh @ V_mh                       # (B, H, S, d_head)
        out = out.transpose(1, 2).reshape(B, S, self.d_model)
        return self.W_O(out), attn_w_mh


#  Demo
torch.manual_seed(42)
mha = AppendixMultiHeadAttention(APPENDIX_D_WORK, APPENDIX_NUM_HEADS)

proj = nn.Linear(APPENDIX_D_MODEL, APPENDIX_D_WORK, bias=False)

# Project the 3D toy embeddings up to the working 16-dim model space (no gradient needed for this demo)
with torch.no_grad():
    x_work = proj(embs).unsqueeze(0)   # (1, 6, 16)

mha_out, head_weights = mha(x_work)
print(f'MHA output shape: {tuple(mha_out.shape)}   head_weights shape: {tuple(head_weights.shape)}')

# Plot each head's attention pattern in its own panel
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for h in range(APPENDIX_NUM_HEADS):
    ax = axes[h]
    w_h = head_weights[0, h].detach().numpy()
    sns.heatmap(w_h, ax=ax, annot=True, fmt='.2f', cmap='Purples',
                xticklabels=APPENDIX_TOKENS, yticklabels=APPENDIX_TOKENS, linewidths=0.5, cbar=False)
    ax.set_title(f'Head {h} attention weights')
    ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)
plt.suptitle('Multi-Head Attention - each head learns a different relationship', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
#  AppendixFeedForward + LayerNorm
# Standard transformer feed-forward block: expand 4x, GELU, project back
class AppendixFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


#  Visualise LayerNorm effect
torch.manual_seed(42)
ffn = AppendixFeedForward(APPENDIX_D_WORK, APPENDIX_D_FF)
norm = nn.LayerNorm(APPENDIX_D_WORK, eps=1e-5)

x_raw = x_work[0]   # (6, 16)

# Run the FFN then LayerNorm without tracking gradients (visualisation only)
with torch.no_grad():
    x_after = ffn(x_raw)       # (6, 16) raw FFN output
    x_normed = norm(x_after)   # (6, 16) after LayerNorm

# Plot each stage's per-token activation profile side by side
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes, [x_raw, x_after, x_normed],
                           ['Input to FFN', 'FFN output (raw)', 'After LayerNorm']):
    data_np = data.detach().numpy()
    for j, token in enumerate(APPENDIX_TOKENS):
        vals = data_np[j]
        ax.plot(vals, alpha=0.7, label=f'{token}  mu={vals.mean():.2f}, s={vals.std():.2f}')
    ax.set_title(title); ax.set_xlabel('Hidden dimension'); ax.set_ylabel('Activation value')
    ax.legend(fontsize=7); ax.axhline(0, color='black', lw=0.5, ls='--')
plt.suptitle('FFN activations before and after LayerNorm', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('LayerNorm centres and normalises each token slice.')
print('Mean and std across dimensions after LN:')
x_normed_np = x_normed.detach().numpy()
for j, token in enumerate(APPENDIX_TOKENS):
    v = x_normed_np[j]
    print(f'  {token:<8}  mean={v.mean():+.4f}  std={v.std():.4f}')


In [ ]:
#  AppendixTransformerBlock
# Pre-LN transformer block: LayerNorm -> MHA -> residual, LayerNorm -> FFN -> residual
class AppendixTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model, eps=1e-5)
        self.mha   = AppendixMultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-5)
        self.ffn   = AppendixFeedForward(d_model, d_ff)

    def forward(self, x, mask=None):

        # Pre-LN attention sublayer, then pre-LN feed-forward sublayer, both with residual adds
        mha_out, attn_w_b = self.mha(self.norm1(x), mask=mask)
        x = x + mha_out
        x = x + self.ffn(self.norm2(x))
        return x, attn_w_b


torch.manual_seed(42)
block1 = AppendixTransformerBlock(APPENDIX_D_WORK, APPENDIX_NUM_HEADS, APPENDIX_D_FF)
block2 = AppendixTransformerBlock(APPENDIX_D_WORK, APPENDIX_NUM_HEADS, APPENDIX_D_FF)

x0 = x_work.clone()   # (1, 6, 16)

# Run two stacked blocks without tracking gradients (inspection only)
with torch.no_grad():
    x1, aw1 = block1(x0)
    x2, aw2 = block2(x1)

print('Input -> Block 1 -> Block 2:')
print(f'  x0: {tuple(x0.shape)}  norm={float(torch.norm(x0)):.3f}')
print(f'  x1: {tuple(x1.shape)}  norm={float(torch.norm(x1)):.3f}')
print(f'  x2: {tuple(x2.shape)}  norm={float(torch.norm(x2)):.3f}')

# Per-token representation norm at each stage of the stack
fig, ax = plt.subplots(figsize=(8, 4))
norms = {
    'Layer 0 (input)': torch.norm(x0[0], dim=-1).detach().numpy(),
    'Layer 1 output':  torch.norm(x1[0], dim=-1).detach().numpy(),
    'Layer 2 output':  torch.norm(x2[0], dim=-1).detach().numpy(),
}
x_pos = np.arange(APPENDIX_SEQ_LEN); width = 0.25

# Plot grouped bars comparing token norms across layers
for k, (label, vals) in enumerate(norms.items()):
    ax.bar(x_pos + k*width, vals, width, label=label, alpha=0.85)
ax.set_xticks(x_pos + width); ax.set_xticklabels(APPENDIX_TOKENS)
ax.set_ylabel('Representation L2 norm')
ax.set_title('Token representations grow through transformer blocks')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
#  AppendixMiniLM - decoder-only transformer language model
class AppendixMiniLM(nn.Module):
    """
    Decoder-only transformer language model.
    Architecture: token_emb -> sinusoidal_PE -> n_layers x AppendixTransformerBlock -> lm_head
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.token_emb = nn.Embedding(vocab_size, d_model)

        # Stack n_layers identical transformer blocks
        self.blocks = nn.ModuleList([
            AppendixTransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.norm_out = nn.LayerNorm(d_model, eps=1e-5)
        pe = appendix_sinusoidal_pe(max_seq, d_model)
        self.register_buffer('pe', pe)

    def forward(self, token_ids, return_attn=False):
        """
        token_ids: (batch, seq_len)  int tensor
        Returns logits: (batch, seq_len, vocab_size)
        """
        S = token_ids.shape[1]
        x = self.token_emb(token_ids) + self.pe[:S]   # (B, S, d_model)

        # Block attention to future positions (decoder-only, causal)
        causal_mask = torch.triu(torch.ones(S, S, dtype=torch.bool), diagonal=1).to(x.device)
        all_attn = []

        # Run each transformer block in turn, collecting per-layer attention weights
        for block in self.blocks:
            x, aw = block(x, mask=causal_mask)
            all_attn.append(aw)
        x = self.norm_out(x)

        # Weight tying: reuse token embedding matrix as output projection
        logits = x @ self.token_emb.weight.T
        if return_attn:
            return logits, all_attn
        return logits


torch.manual_seed(42)
model_demo = AppendixMiniLM(vocab_size=APPENDIX_VOCAB_SIZE, d_model=APPENDIX_D_WORK, n_heads=APPENDIX_NUM_HEADS, d_ff=APPENDIX_D_FF, n_layers=2)

# Run one forward pass to instantiate lazy buffers, without tracking gradients
with torch.no_grad():
    _ = model_demo(torch.tensor([APPENDIX_TOKEN_IDS]))
n_params = sum(p.numel() for p in model_demo.parameters())
print(f'AppendixMiniLM -> {n_params:,} trainable parameters')
for name, p in model_demo.named_parameters():
    print(f'  {name:<40} {tuple(p.shape)}')

In [ ]:
# Minimal state reused by the W_V and scaling comparisons below.
APPENDIX_D_MODEL = appendix_embedding_matrix.shape[1]
APPENDIX_D_HEAD = APPENDIX_D_MODEL
_sentence_embeddings = appendix_embedding_matrix[APPENDIX_TOKEN_IDS]
_attention_scores = _sentence_embeddings @ _sentence_embeddings.T
attn_w = torch.softmax(_attention_scores / math.sqrt(APPENDIX_D_HEAD), dim=-1)
causal_mask = torch.triu(torch.ones(APPENDIX_SEQ_LEN, APPENDIX_SEQ_LEN, dtype=torch.bool), diagonal=1)
print(f"Recap ready: d_model={APPENDIX_D_MODEL}, toy width={APPENDIX_D_WORK}, heads={APPENDIX_NUM_HEADS}")


![Three Transformer architecture families: encoder-only (BERT), decoder-only (GPT), encoder-decoder (T5)](images/transformer-architecture-families.png)

---

## Part 13 - Three Architectures: Reader, Writer, Translator

| Architecture | Mask on self-attn | Primary output | Real examples |
| ------------ | ----------------- | -------------- | ------------- |
| **Encoder-only** | None (bidirectional) | Enriched vector per input token | BERT, RoBERTa |
| **Decoder-only** | Causal (lower-tri) | Next-token logits | GPT, LLaMA |
| **Encoder-Decoder** | Enc: none / Dec: causal | Seq2seq translation | T5, BART |


### 13a - Encoder: The Reader (Bidirectional Attention)

An encoder removes the mask entirely. Every token can attend to every other token. `"cat"` at position 1 can immediately see `"mat"` at position 5.

The encoder does **not** predict next tokens. It produces a sequence of enriched context vectors - one per input token.


> **PyTorch → Keras:** `class AppendixMiniEncoder(nn.Module)` is structurally identical to `AppendixMiniLM` (`nn.Embedding`, `register_buffer`, `nn.ModuleList` of `AppendixTransformerBlock`s) except every block is called with `mask=None` instead of a causal mask, making attention bidirectional. **Keras/TF equivalent:** the same subclassed `tf.keras.layers.Layer`/`Model` pattern as the `AppendixMiniLM` note above, just passing `mask=None` through to each block's `call()` — the mask is the only implementation difference in both frameworks.

In [ ]:
#  Part 13a: AppendixMiniEncoder - decoder-only twin, minus the mask


class AppendixMiniEncoder(nn.Module):
    """
    Encoder-only transformer (BERT-style).
    Passes mask=None to every AppendixTransformerBlock - all tokens see all tokens.
    Output: one enriched d_model-dimensional vector per input token.
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = appendix_sinusoidal_pe(max_seq, d_model)
        self.register_buffer('pe', pe)

        # Stack n_layers identical transformer blocks (bidirectional, since mask=None below)
        self.blocks = nn.ModuleList([
            AppendixTransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.norm_out = nn.LayerNorm(d_model, eps=1e-5)

    def forward(self, token_ids, return_attn=False):
        S = token_ids.shape[1]
        x = self.token_emb(token_ids) + self.pe[:S]
        all_attn = []
        for block in self.blocks:
            x, aw = block(x, mask=None)   # None = bidirectional; the only difference
            all_attn.append(aw)
        out = self.norm_out(x)
        if return_attn:
            return out, all_attn
        return out


torch.manual_seed(42)
encoder = AppendixMiniEncoder(APPENDIX_VOCAB_SIZE, APPENDIX_D_WORK, APPENDIX_NUM_HEADS, APPENDIX_D_FF, n_layers=2)
ids_batch = torch.tensor([APPENDIX_TOKEN_IDS])

# Run the encoder forward pass without tracking gradients
with torch.no_grad():
    enc_out, enc_attns = encoder(ids_batch, return_attn=True)

print(f'Encoder output shape: {tuple(enc_out.shape)}')
print(f'  -> One enriched {APPENDIX_D_WORK}-dim vector per token (same shape as decoder output)')
print(f'  -> These are NOT next-token predictions - they are context carriers')
print()
print('The entire AppendixMiniEncoder class differs from AppendixMiniLM in exactly one place:')
print('  AppendixMiniLM      block(x, mask=causal_mask)   # upper triangle -> -inf')
print('  AppendixMiniEncoder block(x, mask=None)           # nothing blocked')

#### Predict first - which cells in the heatmap open up?

The decoder's causal heatmap has a hard lower-triangle: `"the"` at [0] attends only to itself.

Now we set `mask=None`. **Predict:**
- How many tokens will `"the"` at [0] attend to now - 1, 3, or 6?
- Will `"the"` at [0] have a non-zero score for `"mat"` at position [5]?


> **PyTorch → Keras:** reuses one `AppendixMultiHeadAttention` module instance and calls it twice — once with `mask=None`, once with a `torch.triu(...)` causal mask — under `torch.no_grad()` to compare bidirectional vs. causal attention with identical weights. **Keras/TF equivalent:** the same `tf.keras.layers.Layer` instance called twice with different `mask` arguments (`training=False` in place of `no_grad()`); calling one layer instance twice with different inputs/masks works identically in both frameworks since weights live on the layer, not the call.

In [ ]:
#  Encoder vs Decoder: SAME MHA weights, SAME input - only the mask differs
torch.manual_seed(42)
shared_mha = AppendixMultiHeadAttention(APPENDIX_D_WORK, APPENDIX_NUM_HEADS)
causal_mask_cmp = torch.triu(torch.ones(APPENDIX_SEQ_LEN, APPENDIX_SEQ_LEN, dtype=torch.bool), diagonal=1)

# Call the SAME MHA weights twice, once with no mask and once with the causal mask
with torch.no_grad():
    _, w_bidir      = shared_mha(x_work, mask=None)
    _, w_causal_cmp = shared_mha(x_work, mask=causal_mask_cmp)

bidir_h0  = w_bidir[0, 0].detach().numpy()
causal_h0 = w_causal_cmp[0, 0].detach().numpy()

# Plot head-0 attention side by side: bidirectional vs causal
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, data, title, cmap in [
    (axes[0], bidir_h0,  'Encoder  (mask=None, bidirectional)\n'
              '"the" at [0] already attends to "cat", "sat", "mat" in layer 1', 'Greens'),
    (axes[1], causal_h0, 'Decoder  (mask=causal_mask, lower triangle only)\n'
              '"the" at [0] sees only itself', 'Oranges'),
]:
    sns.heatmap(data, ax=ax, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=APPENDIX_TOKENS, yticklabels=APPENDIX_TOKENS, linewidths=0.5,
                cbar_kws={'label': 'attention weight'})
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)

plt.suptitle('Same MHA layer, same weights, same input - the mask is the ONLY difference\n'
             'This is the complete implementation difference between Encoder and Decoder',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('Encoder: "the" at [0] attends to all 6 tokens from layer 1.')
print('Decoder: "the" at [0] sees only itself.')
print()
print('Consequence:')
print('  Encoder -> each output holds full bidirectional context')
print('  Decoder -> each output depends ONLY on what came before')


> **PyTorch → Keras:** toggles `USE_ENCODER_MASK` to switch between `mask=None` and the cached causal mask on the same `shared_mha` module inside `torch.no_grad()`. **Keras/TF equivalent:** the identical toggle pattern on a `tf.keras.layers.Layer` instance, calling it with `training=False` — no framework-specific difference beyond the no-grad/training-flag naming.

In [ ]:
#  EXERCISE 4 - toggle the mask and watch the heatmap change
# Flip USE_ENCODER_MASK to False.
# PREDICT first: which cells in row 0 will go to zero?
USE_ENCODER_MASK = True   # set to False to turn the encoder into a decoder

# Pick no mask (encoder) or the causal mask (decoder) based on the toggle
mask_ex4 = None if USE_ENCODER_MASK else causal_mask_cmp

# Recompute head-0 attention under the selected mask
with torch.no_grad():
    _, w_ex4 = shared_mha(x_work, mask=mask_ex4)
w_ex4_h0 = w_ex4[0, 0].detach().numpy()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
mode_label = 'Encoder (mask=None)' if USE_ENCODER_MASK else 'Decoder (causal mask)'
sns.heatmap(w_ex4_h0, ax=ax, annot=True, fmt='.2f',
            cmap='Greens' if USE_ENCODER_MASK else 'Oranges',
            xticklabels=APPENDIX_TOKENS, yticklabels=APPENDIX_TOKENS, linewidths=0.5, cbar=False)
ax.set_title(f'Head 0 attention - {mode_label}')
ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()
print(f'Mode: {mode_label}')


#### But wait - why not just pass the encoder output directly as the decoder's starting state?

There are two problems:

**Problem 1 - The causal mask cuts off the source.** The decoder still applies its causal mask.

**Problem 2 - Source and target have different lengths.** You can't stack them as positions.

The solution is **Cross-Attention**: the decoder queries the encoder output as a separate "table" at every decoding step.


### 13b - Cross-Attention: The Bridge

In cross-attention:

$$Q = \text{decoder state} \cdot W_Q \qquad K, V = \text{encoder output} \cdot W_K, W_V$$

The decoder asks: *"Given what I have generated so far ($Q$), which part of the source text ($K$) is most relevant, and what should I extract from it ($V$)?"*


> **PyTorch → Keras:** `class AppendixCrossAttention(nn.Module)` mirrors `AppendixMultiHeadAttention` but takes two inputs — `decoder_x` for `W_Q` and `encoder_kv` for `W_K`/`W_V` — with no mask applied on the source (encoder) dimension. **Keras/TF equivalent:** a `tf.keras.layers.Layer` subclass whose `call(self, decoder_x, encoder_kv)` takes two arguments and projects each through separate `Dense` layers — Keras also ships a built-in `tf.keras.layers.AppendixMultiHeadAttention(num_heads, key_dim)` that natively supports this two-input (query, value) cross-attention pattern out of the box.

In [ ]:
#  AppendixCrossAttention: Q from decoder, K/V from encoder


class AppendixCrossAttention(nn.Module):
    """
    Cross-attention layer.
    Q  <- decoder's current state
    K  <- encoder's output
    V  <- encoder's output
    No mask on the encoder dimension - the decoder can attend to ANY source position.
    """

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, decoder_x, encoder_kv):
        """
        decoder_x : (B, tgt_len, d_model)
        encoder_kv: (B, src_len, d_model)
        Score matrix: (B, heads, tgt_len, src_len)  - no mask applied.
        """
        B, tgt_len, _ = decoder_x.shape
        src_len = encoder_kv.shape[1]

        # Project decoder query and encoder key/value, then split each into per-head slices
        Q_ca = self.W_Q(decoder_x).reshape(B, tgt_len, self.n_heads, self.d_head).transpose(1, 2)
        K_ca = self.W_K(encoder_kv).reshape(B, src_len, self.n_heads, self.d_head).transpose(1, 2)
        V_ca = self.W_V(encoder_kv).reshape(B, src_len, self.n_heads, self.d_head).transpose(1, 2)
        scores = (Q_ca @ K_ca.transpose(-2, -1)) / math.sqrt(self.d_head)
        cross_w = torch.softmax(scores, dim=-1)   # (B, H, tgt, src)
        out = cross_w @ V_ca

        # Merge heads back into a single d_model-dimensional vector per decoder position
        out = out.transpose(1, 2).reshape(B, tgt_len, self.d_model)
        return self.W_O(out), cross_w


#  Quick demo: one decoder query token attending to six encoder positions
torch.manual_seed(5)
cross_demo = AppendixCrossAttention(APPENDIX_D_WORK, APPENDIX_NUM_HEADS)

# Use the encoder's first-token output as a stand-in single decoder query
dec_state_demo = enc_out[:, :1, :]

# Run cross-attention without tracking gradients
with torch.no_grad():
    ca_out_demo, ca_w_demo = cross_demo(dec_state_demo, enc_out)

print('Cross-attention shapes:')
print(f'  Decoder query  : {tuple(dec_state_demo.shape)}  (1 decoder token)')
print(f'  Encoder K/V    : {tuple(enc_out.shape)}  (6 source tokens)')
print(f'  Score matrix   : {tuple(ca_w_demo.shape)}')
print()
print('The single decoder query scored all 6 encoder positions.')
print('Which source token it attends to is entirely learned by the loss.')

### 13c - Encoder-Decoder: The Translator

A full encoder-decoder wires both halves together with cross-attention in every decoder block. We train it on a toy task - **reversing a three-word phrase** - to force cross-attention to do real work.

```
Encoder: ["the", "cat", "sat"]  ->  [ enriched vectors: V_the, V_cat, V_sat ]
Decoder: [<BOS>]  -> "sat" ;  [<BOS>, sat]  -> "cat" ;  [<BOS>, sat, cat]  -> "the"
```


> **PyTorch → Keras:** `AppendixDecoderBlockWithCrossAttn` composes three `nn.Module` sub-layers (causal self-attention, `AppendixCrossAttention`, `AppendixFeedForward`) each behind its own `nn.LayerNorm`; `AppendixMiniEncoderDecoder` wires an `nn.ModuleList` encoder stack and an `nn.ModuleList` decoder stack together, with a standalone (untied) `nn.Linear` `lm_head` rather than the weight-tying `AppendixMiniLM` used. **Keras/TF equivalent:** the same three-sub-layer composition as `tf.keras.layers.Layer`s under separate `LayerNormalization` instances, `encode()`/full `call()` split into two methods as needed, and `tf.keras.layers.Dense(vocab_size)` for the untied output head.

In [ ]:
#  AppendixDecoderBlockWithCrossAttn + AppendixMiniEncoderDecoder


class AppendixDecoderBlockWithCrossAttn(nn.Module):
    """
    Full decoder block - three sub-layers:
      1. Causal self-attention  - the decoder looks at its own generated tokens
      2. Cross-attention        - the decoder queries the encoder's source map
      3. Feed-forward           - per-token nonlinear transformation
    """

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model, eps=1e-5)
        self.self_attn = AppendixMultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-5)
        self.cross_attn = AppendixCrossAttention(d_model, n_heads)
        self.norm3 = nn.LayerNorm(d_model, eps=1e-5)
        self.ffn = AppendixFeedForward(d_model, d_ff)

    def forward(self, x, encoder_output, causal_mask=None):
        sa_out, sa_w = self.self_attn(self.norm1(x), mask=causal_mask)
        x = x + sa_out
        ca_out, ca_w = self.cross_attn(self.norm2(x), encoder_output)
        x = x + ca_out
        x = x + self.ffn(self.norm3(x))
        return x, sa_w, ca_w


class AppendixMiniEncoderDecoder(nn.Module):
    """
    Encoder-Decoder transformer (T5 / original-Transformer style).
    Encoder : bidirectional - produces a frozen source map (K, V for cross-attn)
    Decoder : causal self-attn + cross-attn at every block - LM head
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.d_model = d_model
        self.emb = nn.Embedding(vocab_size, d_model)
        pe = appendix_sinusoidal_pe(max_seq, d_model)
        self.register_buffer('pe', pe)

        # Stack of bidirectional encoder blocks (mask=None in .encode() below)
        self.enc_blocks = nn.ModuleList([
            AppendixTransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.enc_norm = nn.LayerNorm(d_model, eps=1e-5)

        # Stack of decoder blocks, each with causal self-attention + cross-attention
        self.dec_blocks = nn.ModuleList([
            AppendixDecoderBlockWithCrossAttn(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.dec_norm = nn.LayerNorm(d_model, eps=1e-5)
        self.lm_head = nn.Linear(d_model, vocab_size)

    # Encode the source sequence bidirectionally into a fixed context map
    def encode(self, src_ids):
        S_enc = src_ids.shape[1]
        x = self.emb(src_ids) + self.pe[:S_enc]
        for block in self.enc_blocks:
            x, _ = block(x, mask=None)
        return self.enc_norm(x)

    def forward(self, src_ids, tgt_ids):
        """
        src_ids: (B, src_len) - what the encoder reads
        tgt_ids: (B, tgt_len) - decoder input shifted right (teacher-forced)
        Returns logits (B, tgt_len, vocab_size) and all cross-attn weight tensors.
        """
        enc_out_s2s = self.encode(src_ids)
        T_dec = tgt_ids.shape[1]
        x = self.emb(tgt_ids) + self.pe[:T_dec]
        causal_mask_s2s = torch.triu(torch.ones(T_dec, T_dec, dtype=torch.bool), diagonal=1)
        causal_mask_s2s = causal_mask_s2s.to(x.device)
        all_ca_w = []

        # Run each decoder block: causal self-attention, then cross-attention into the encoder map
        for block in self.dec_blocks:
            x, _, ca_w = block(x, enc_out_s2s, causal_mask=causal_mask_s2s)
            all_ca_w.append(ca_w)
        x = self.dec_norm(x)
        logits = self.lm_head(x)
        return logits, all_ca_w


torch.manual_seed(42)
seq2seq_demo = AppendixMiniEncoderDecoder(APPENDIX_VOCAB_SIZE, APPENDIX_D_WORK, APPENDIX_NUM_HEADS, APPENDIX_D_FF, n_layers=2)
print('AppendixMiniEncoderDecoder architecture:')
print(f'  Encoder: {len(seq2seq_demo.enc_blocks)} x AppendixTransformerBlock  (mask=None, bidirectional)')
print(f'  Decoder: {len(seq2seq_demo.dec_blocks)} x AppendixDecoderBlockWithCrossAttn')
print(f'           -> self-attn (causal) + cross-attn + FFN')
print(f'  Shared vocab: {APPENDIX_VOCAB_SIZE} tokens')

> **PyTorch → Keras:** `build_rev_batch` pads variable-length id lists and wraps them with `torch.tensor(srcs, dtype=torch.long)` (long dtype required for embedding-table indices). **Keras/TF equivalent:** `tf.constant(srcs, dtype=tf.int64)` or `tf.keras.utils.pad_sequences(...)` for the padding step — `tf.keras.layers.Embedding` also accepts plain `int32`/`int64` id tensors the same way `nn.Embedding` does.

In [ ]:
#  Toy training data: reverse a 3-word phrase
# Source:  ["the", "cat", "sat"]
# Target:  ["sat", "cat", "the", "<EOS>"]  (decoder input = ["<BOS>"] + target[:-1])

BOS_ID, EOS_ID = APPENDIX_VOCAB['<BOS>'], APPENDIX_VOCAB['<EOS>']

source_phrases_rev = [
    ['the', 'cat', 'sat'], ['the', 'dog', 'ran'],
    ['a',   'big', 'cat'], ['the', 'cat', 'ran'],
    ['a',   'dog', 'sat'], ['the', 'big', 'dog'],
]

# Build (source, decoder-input, decoder-target) triples for the reversal task
REV_DATA = []
for phrase in source_phrases_rev:
    src_ids_r = [APPENDIX_VOCAB[w] for w in phrase]
    rev_ids   = [APPENDIX_VOCAB[w] for w in reversed(phrase)]
    tgt_in_r  = [BOS_ID] + rev_ids
    tgt_out_r = rev_ids + [EOS_ID]
    REV_DATA.append((src_ids_r, tgt_in_r, tgt_out_r))

print(f'Reversal task - {len(REV_DATA)} training pairs:')
for src_r, _, tout_r in REV_DATA[:3]:
    print(f'  {[APPENDIX_IDX2WORD[i] for i in src_r]}  ->  {[APPENDIX_IDX2WORD[i] for i in tout_r]}')
print('  ...')


# Right-pad source/target sequences to the batch max length, then stack into tensors
def build_rev_batch(data, pad_id=0):
    src_max = max(len(s) for s, _, _ in data)
    tgt_max = max(len(t) for _, t, _ in data)
    srcs, tins, touts = [], [], []
    for s, ti, to in data:
        srcs.append(s  + [pad_id] * (src_max - len(s)))
        tins.append(ti + [pad_id] * (tgt_max - len(ti)))
        touts.append(to + [pad_id] * (tgt_max - len(to)))
    return (torch.tensor(srcs,  dtype=torch.long),
            torch.tensor(tins,  dtype=torch.long),
            torch.tensor(touts, dtype=torch.long))


src_rev, tgt_in_rev, tgt_out_rev = build_rev_batch(REV_DATA)
print(f'Batch shapes - src: {tuple(src_rev.shape)}  '
      f'tgt_in: {tuple(tgt_in_rev.shape)}  tgt_out: {tuple(tgt_out_rev.shape)}')


> **PyTorch → Keras:** the training loop reuses the `optimizer.zero_grad()` / `loss.backward()` / `clip_grad_norm_` / `optimizer.step()` pattern from the `AppendixMiniLM` training loop, but with `nn.CrossEntropyLoss(ignore_index=0)` so padded target positions (id 0) don't contribute to the loss, and `torch.argmax(logits_s2s, dim=-1)` for greedy decoding accuracy. **Keras/TF equivalent:** the same `tf.GradientTape` loop as before; the padding-aware loss becomes `tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')` combined with a boolean mask (`tgt_out_rev != 0`) applied before averaging, since Keras' losses don't have a built-in `ignore_index` argument; `tf.argmax(logits_s2s, axis=-1)` mirrors `torch.argmax`.

In [ ]:
#  Train the encoder-decoder on the reversal task
torch.manual_seed(42)
seq2seq = AppendixMiniEncoderDecoder(APPENDIX_VOCAB_SIZE, APPENDIX_D_WORK, APPENDIX_NUM_HEADS, APPENDIX_D_FF, n_layers=2)
optimizer_s2s = torch.optim.Adam(seq2seq.parameters(), lr=5e-3)
loss_fn_s2s = nn.CrossEntropyLoss(ignore_index=0)

EPOCHS_S2S = 500
s2s_loss_h, s2s_acc_h = [], []

# Standard training loop with a padding-aware cross-entropy loss (ignore_index=0)
for epoch in range(EPOCHS_S2S):
    seq2seq.train()
    optimizer_s2s.zero_grad()
    logits_s2s, _ = seq2seq(src_rev, tgt_in_rev)
    loss_s2s = loss_fn_s2s(logits_s2s.reshape(-1, APPENDIX_VOCAB_SIZE), tgt_out_rev.reshape(-1))
    loss_s2s.backward()
    torch.nn.utils.clip_grad_norm_(seq2seq.parameters(), 1.0)
    optimizer_s2s.step()

    # Periodically compute padding-aware accuracy and log progress
    if (epoch + 1) % 25 == 0:
        with torch.no_grad():
            preds_s2s = torch.argmax(logits_s2s, dim=-1)
            pad_mask_s2s = (tgt_out_rev != 0)
            correct = (preds_s2s == tgt_out_rev) & pad_mask_s2s
            acc_s2s = correct.float().sum().item() / pad_mask_s2s.float().sum().item()
        s2s_loss_h.append(loss_s2s.item())
        s2s_acc_h.append(acc_s2s)
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch+1:4d}  loss={loss_s2s.item():.4f}  acc={acc_s2s:.2%}')

print('\nTraining complete.  Greedy decoding results:')
seq2seq.eval()

# Greedily decode each example and compare against the gold reversal
for src_r, tin_r, tout_r in REV_DATA:
    with torch.no_grad():
        lgts, _ = seq2seq(
            torch.tensor([src_r],  dtype=torch.long),
            torch.tensor([tin_r],  dtype=torch.long),
        )
    pred_ids = torch.argmax(lgts[0], dim=-1).tolist()
    src_w  = [APPENDIX_IDX2WORD[i] for i in src_r]
    pred_w = [APPENDIX_IDX2WORD[i] for i in pred_ids]
    gold_w = [APPENDIX_IDX2WORD[i] for i in tout_r]
    ok = '[ok]' if pred_ids == tout_r else '[xx]'
    print(f'  {ok}  source={src_w}  pred={pred_w}  gold={gold_w}')


#### Predict first - draw the cross-attention map

Source: `["the", "cat", "sat"]` -> reversed target: `["sat", "cat", "the"]`.

| Decoder step | Generating | Highest source attention should be at? |
| ------------ | ---------- | --------------------------------------- |
| Step 0 (`<BOS>` -> "sat") | "sat" | source[ ? ] |
| Step 1 (`"sat"` -> "cat") | "cat" | source[ ? ] |
| Step 2 (`"sat cat"` -> "the") | "the" | source[ ? ] |


> **PyTorch → Keras:** runs the trained `seq2seq` model under `torch.no_grad()`, then `ca_w[0].mean(dim=0).detach().numpy()` averages the cross-attention weights over heads for plotting. **Keras/TF equivalent:** call the model with `training=False`, then `tf.reduce_mean(ca_w[0], axis=0).numpy()` — same "average over the head axis" reduction, `axis=` instead of `dim=`.

In [ ]:
#  Cross-attention heatmap: decoder reading the encoder blueprint
test_src_s2s = [APPENDIX_VOCAB['the'], APPENDIX_VOCAB['cat'], APPENDIX_VOCAB['sat']]
test_tgt_s2s = [BOS_ID, APPENDIX_VOCAB['sat'], APPENDIX_VOCAB['cat']]

seq2seq.eval()

# Run the trained model once more, forward-only, to capture cross-attention weights
with torch.no_grad():
    logits_vis, ca_vis = seq2seq(
        torch.tensor([test_src_s2s], dtype=torch.long),
        torch.tensor([test_tgt_s2s], dtype=torch.long),
    )

src_lbls = [APPENDIX_IDX2WORD[i] for i in test_src_s2s]
dec_lbls = ['<BOS>->sat', 'sat->cat', 'cat->the']
n_dec_layers = len(seq2seq.dec_blocks)
fig, axes = plt.subplots(1, n_dec_layers, figsize=(6 * n_dec_layers, 4.2))

# plt.subplots returns a bare Axes (not an array) when there's only one column
if n_dec_layers == 1:
    axes = [axes]

# Plot each decoder layer's cross-attention map, averaged over heads
for layer_i, (ax, ca_w) in enumerate(zip(axes, ca_vis)):
    mean_ca = ca_w[0].mean(dim=0).detach().numpy()   # avg heads -> (tgt, src)
    sns.heatmap(mean_ca, ax=ax, annot=True, fmt='.2f',
                cmap='YlOrRd', xticklabels=src_lbls, yticklabels=dec_lbls,
                linewidths=0.6, vmin=0, vmax=1, cbar_kws={'label': 'cross-attn weight'})
    ax.set_title(f'Decoder layer {layer_i}  (mean over {APPENDIX_NUM_HEADS} heads)')
    ax.set_xlabel('Source position (encoder output)')
    ax.set_ylabel('Decoder step -> predicted token')
    ax.tick_params(axis='x', rotation=0)

plt.suptitle(f'Cross-attention: decoder querying the encoder  |  source: {src_lbls}',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('If reversal is learned, row 0 (->"sat") should attend most to source[2]="sat".')
print('Row 1 (->"cat") should attend most to source[1]="cat".  Etc.')
print('The cross-attention map IS the learned "look at the right source position" rule.')


### 13d - Why the Industry Moved to Decoder-Only

| Issue | Encoder-Decoder | Decoder-Only |
| ----- | --------------- | ------------ |
| Training data | Needs paired input/output sequences | Eats *any* raw text |
| KV cache at inference | Two separate caches | One growing cache |
| Information routing | Must compress source through cross-attention | Implicit in early layers |
| Scaling | Complex orchestration | Simple stack |

#### What is a "KV cache", actually?

The autoregressive loop in Part 9 (`generate_next`) re-encodes the entire growing context from scratch at every new token, recomputing K and V for every earlier position again even though those tokens haven't changed. A KV cache is the optimisation of not redoing that work: every past token's K and V vectors are computed once and stored, so each new step only computes Q/K/V for the one new token and reuses the cached K/V for everything before it. An encoder-decoder model needs two such caches (one for the fixed encoder output, one for the growing decoder prefix); a decoder-only model needs only one (the growing prefix) - exactly the "two caches vs. one growing cache" row above.

Not implemented here: the toy generation loop is short enough (a handful of tokens) that recomputation is free. A real inference server needs the cache so that per-token latency stays roughly constant instead of growing with sequence length.


> **PyTorch → Keras:** instantiates all three architectures under `torch.no_grad()` forward passes and sums `p.numel() for p in model.parameters()` to count trainable parameters. **Keras/TF equivalent:** call each model with `training=False`, then `model.count_params()` (or `sum(tf.size(w).numpy() for w in model.trainable_variables)`) in place of `.numel()`/`.parameters()`.

In [ ]:
#  Architecture comparison: parameter cost and capability table
torch.manual_seed(0)

_enc_cmp    = AppendixMiniEncoder(APPENDIX_VOCAB_SIZE, APPENDIX_D_WORK, APPENDIX_NUM_HEADS, APPENDIX_D_FF, n_layers=2)
_dec_cmp    = AppendixMiniLM(APPENDIX_VOCAB_SIZE, APPENDIX_D_WORK, APPENDIX_NUM_HEADS, APPENDIX_D_FF, n_layers=2)
_encdec_cmp = AppendixMiniEncoderDecoder(APPENDIX_VOCAB_SIZE, APPENDIX_D_WORK, APPENDIX_NUM_HEADS, APPENDIX_D_FF, n_layers=2)

_dummy6 = torch.tensor([APPENDIX_TOKEN_IDS])
_dummy3 = torch.tensor([APPENDIX_TOKEN_IDS[:3]])

# Instantiate lazy state for all three architectures with one forward pass each
with torch.no_grad():
    _enc_cmp(_dummy6)
    _dec_cmp(_dummy6)
    _encdec_cmp(_dummy6, _dummy3)

p_enc_cmp    = sum(p.numel() for p in _enc_cmp.parameters())
p_dec_cmp    = sum(p.numel() for p in _dec_cmp.parameters())
p_encdec_cmp = sum(p.numel() for p in _encdec_cmp.parameters())

rows_cmp = [
    ('',                'Encoder-Only',  'Decoder-Only',     'Encoder-Decoder'),
    ('Self-attn mask',  'None (bidir)',  'Causal',           'Enc: none / Dec: causal'),
    ('Cross-attn',      'No',            'No',               f'{len(_encdec_cmp.dec_blocks)} layers'),
    ('Parameters',      f'{p_enc_cmp:,}', f'{p_dec_cmp:,}', f'{p_encdec_cmp:,}'),
    ('Training data',   'Labelled corpus', 'Any raw text',  'Paired sequences'),
    ('Real examples',   'BERT, RoBERTa',   'GPT, LLaMA',    'T5, BART'),
]

col_w = [20, 18, 20, 26]
sep = '  ' + '-' * (sum(col_w) + 6)
print(sep)
for i, row in enumerate(rows_cmp):
    line = '  ' + '  '.join(f'{c:<{w}}' for c, w in zip(row, col_w))
    print(line)
    if i == 0:
        print(sep)
print(sep)
print()
overhead = p_encdec_cmp - p_dec_cmp
print(f'Cross-attention overhead : {overhead:+,} params  '
      f'(+{overhead/p_dec_cmp*100:.1f}% vs decoder-only at same depth/width)')
print()
print('At GPT-3 scale (175B parameters), that overhead is non-trivial -')
print('one reason the industry consolidated around decoder-only for general LLMs.')


---

## What This Notebook Covered (and What It Didn't)

Before the completed roadmap below, here is every technique named anywhere in this notebook, sorted
into the tier its actual treatment earns.

### Tier 1 — Implemented and Demonstrated

- **Bidirectional encoder self-attention** (`mask=None`) — proven against causal attention with a
  shared-weights, same-input heatmap comparison (Part 2).
- **Causal decoder self-attention** (`mask=causal_mask`) — same comparison, opposite mask.
- **Cross-attention** (Q=decoder, K=V=encoder, asymmetric `(T×S)` score matrix) — built as its own
  class, shape-verified with `T != S`, and visualised as a trained attention map (Parts 4 and 6).
- **The pre-attention seq2seq bottleneck** — measured directly via cosine similarity between
  per-position and mean-pooled encoder vectors on two different source sequences (Part 3).
- **Teacher-forced training** on the reversal task, with real loss curves and a measured validation
  sequence accuracy (Part 5).
- **Cross-attention's learned routing** — the anti-diagonal attention pattern is measured (mean
  weight on the anti-diagonal, not asserted) and animated frame-by-frame (Part 6).
- **Free-running (greedy) autoregressive generation** — a real `greedy_decode` loop that never sees
  the gold target, with free-running accuracy measured against the teacher-forced number above and
  the gap explained as exposure bias (Part 6a).
- **Toy → real bridge** — a live `t5-small` summarisation call (guarded by a graceful fallback if
  offline), plus the full toy→production hyperparameter mapping table (Part 7).

### Tier 2 — Explained but Not Fully Implemented

- **Beam search decoding** — a from-scratch, illustrative `beam_search_decode` (top-`k` partial
  sequences by cumulative log-probability) runs on the trained model and is compared to greedy
  output, but it skips length normalisation, batched beam execution, and per-beam EOS handling that
  `model.generate(num_beams=k)` provides in production (Part 6b). Reason: the point here is _why_
  tracking more than one candidate can beat greedy, not shipping a production-grade decoder.
- **T5 span-corruption / BART denoising pre-training objectives** — explained in plain English
  (what each corruption/reconstruction scheme does and why it lets the model learn from unlabelled
  text) immediately after the toy→real parameter table, but not built (Part 7). Reason: this
  notebook's reversal task is fully supervised by design, specifically to keep the focus on
  cross-attention mechanics rather than large-scale self-supervised pre-training.
- **Weight tying between the token embedding and the LM head** — named as the reason `lm_head` uses
  `bias=False` (a convention this notebook follows), but the embedding and `lm_head` weights are
  never actually tied in code (Part 5). Reason: tying is a parameter-saving optimisation aimed at
  production-scale vocabularies; at `vocab_size=13` the savings are negligible.
- **Padding masks for variable-length batches** — `nn.CrossEntropyLoss(ignore_index=PAD)` is real,
  running code, but every source/target sequence in this notebook is a fixed `SEQ_LEN=4`, so no
  batch actually contains a padded position — the mechanism has never been exercised on real
  variable-length input (Part 5). Reason: reversal's fixed length keeps every visualisation the same
  shape end to end; a real translation task would need this exercised for real.

### Tier 3 — Named but Out of Scope

- **KV-caching across generation steps** — the production technique that avoids recomputing past
  decoder keys/values at every generation step. Out of scope: `greedy_decode` above already gets its
  speed from encoding once; caching decoder K/V across steps is a real optimisation this notebook
  doesn't need at `SEQ_LEN=4` to make its point.
- **Pre-attention RNN encoder-decoders and Bahdanau/Luong attention** — the historical bridge
  between the fixed-vector bottleneck (Part 3) and modern cross-attention. Out of scope: covering
  the actual RNN recurrence and the original additive/multiplicative attention formulas would double
  this notebook's scope for a point that's about the _bottleneck_, which is already measured
  directly.
- **Multilingual encoder-decoders (mT5, mBART, NLLB)** — same architecture as T5/BART, scaled to
  100+ languages with a shared multilingual vocabulary. Out of scope: no new architectural idea over
  what Part 7 already bridges to.
- **Machine translation and question-answering as applications** — encoder-decoder's other two
  headline use-cases beyond the summarisation demoed live in Part 7. Out of scope: the underlying
  mechanism (cross-attention over an encoded source) is identical across all three; only the
  training data changes.
- **Why decoder-only architectures (GPT-3/4, LLaMA) came to dominate general-purpose LLMs despite
  encoder-decoder's structural fit for seq2seq tasks** — a real, actively-discussed design-space
  question (compute/deployment trade-offs, one model serving every task). Out of scope: it's a
  systems/product question that sits a level above this notebook's architecture-mechanics focus.


---

## Summary — Completed Roadmap

| Step | Part | Concept | Key Idea |
|------|------|---------|----------|
| 1 | The Contract | What encoder-decoder solves | Bidirectionality + variable-length I/O |
| 2 | The Encoder | Bidirectional attention | mask=None gives every token a 360-degree view |
| 3 | The Bottleneck | Why naive pooling fails | Fixed vector loses positional detail at scale |
| 4 | Cross-Attention | The bridge | Q = decoder, K = V = encoder; asymmetric $(T \times S)$ scores |
| 5 | Full Model + Training | Wire all components | Teacher-forced seq2seq converges on reversal |
| 6 | Cross-Attention Map | Visualise learned routing | Anti-diagonal confirms decoder step $i$ -> source $S-1-i$ |
| 6a | Free-Running Decoding | Free-running greedy decode (+ beam-search aside) | Teacher-forced != free-running; exposure bias measured, not assumed |
| 7 | Toy to Real | T5 / BART mapping | Same architecture, wider vectors, larger vocab |

---

### Key insights to keep

- The **only** code difference between an encoder block and a decoder block is
  `mask=None` vs `mask=causal_mask` — one argument controls bidirectionality.
- Cross-attention score matrix shape is $(T_{\text{tgt}} \times S_{\text{src}})$ —
  **asymmetric**, unlike self-attention's $(S \times S)$. No mask on the encoder side.
- The information bottleneck in pre-attention seq2seq came from collapsing the source
  into a fixed vector. Cross-attention replaces it with $S$ live source vectors — the
  capacity grows linearly with source length.
- Held-out sequence accuracy is evidence that the model learned the reversal task beyond its
  training examples; the anti-diagonal heatmap separately diagnoses routing consistent with
  that rule. Attention weights alone do not establish a causal explanation.
- T5-small has 60 M parameters and $d_{\text{model}}=512$; our toy has ~20 K and
  $d_{\text{model}}=32$. The cross-attention formula $Q(K^\top)/\sqrt{d_k}$ is
  unchanged — just wider vectors.


---

## Series Epilogue: Preserved Coverage Ledger

The next two cells are the original all-in-one notebook's coverage ledger and complete-journey summary. The following preserved ledger and roadmap summarize the complete three-notebook Transformer Foundations series.


---

## What the Transformer Foundations Series Covered (and What It Didn't)

Before the final recap below, here is the honest three-tier breakdown of the full topic space a complete "how Transformers work" treatment would include, and exactly where this series's actual code lands on each one.

**Tier 1 - Implemented and demonstrated:** essentially everything above, Parts 1-14 - see the completed roadmap table in the Summary immediately below for the full list.

**Tier 2 - Explained but not fully implemented** (accurate explanation, no full toy-then-real demo):

- **KV-caching** - explained in plain English right after the encoder-decoder vs. decoder-only comparison table, tied directly to the recomputation this series's own `generate_next` loop does at every step, but no actual cache tensors are built.
- **Pre-LN vs. Post-LN placement** - this series uses and names Pre-LN; Post-LN is named as the historical alternative but not built side-by-side for comparison.

**Tier 3 - Named but out of scope** (acknowledged, with a one-line reason):

- **Learned absolute positional embeddings** and **ALiBi** - two more positional-encoding schemes in the same family as sinusoidal PE / RoPE, omitted to keep the sinusoidal -> RoPE arc as the notebook's throughline. (DistilGPT-2's real `wpe` table is inspected in Part 14 as a concrete example of the learned variant, without re-deriving it from scratch.)
- **Dropout** - a standard regulariser, omitted because the toy models are never at risk of overfitting a six-sentence corpus.
- **Model scaling laws** - a training-economics topic (compute/data/params trade-offs), orthogonal to the architecture mechanics this series focuses on.
- **Nucleus (top-p) sampling** - greedy, temperature, top-k, and the Part 3 beam-search aside already demonstrate the decoding-strategy design space.
- **Efficient-attention variants** (sliding-window, sparse attention, FlashAttention, multi-query / grouped-query attention) - production-scale engineering optimisations of the exact O(n^2) attention already built here, not a different mechanism.
- **Explicit padding/attention masks for batched variable-length input** - this series's training loops use fixed-length batches or loss-side masking (`ignore_index`); a production implementation would also mask padded positions out of attention itself.


---

## Series Summary - The Complete Transformer Journey

Across three notebooks, we've traced every component from raw words to generated tokens:

| Step | Component | What happens |
| ---- | --------- | ------------ |
| 1  | **Tokeniser** | Text -> integer IDs |
| 2  | **Token Embedding** | IDs -> dense vectors in semantic space |
| 3  | **Positional Encoding** | Add position signal (sin/cos or RoPE rotation) |
| 4  | **Q/K/V Projection** | Three learned views of each token vector |
| 5  | **Scaled Dot-Product Attention** | Soft dictionary lookup - compute relevance scores |
| 6  | **Multi-Head Attention** | H parallel attention views concatenated |
| 7  | **Feed-Forward Network** | Per-token nonlinear transformation |
| 8  | **LayerNorm + Residuals** | Stabilise activations, guarantee gradient flow |
| 9  | **Repeat x L** | Stack L transformer blocks |
| 10 | **LM Head** | Project to vocabulary -> logits -> softmax -> distribution |
| 11 | **W_V Relevance Filter** | W_V extracts the task-specific payload |
| 12 | **Causal Triangle** | Position n accumulates n+1 tokens; depth chains richness |
| 13 | **Encoder Architecture** | mask=None gives bidirectional context |
| 14 | **Cross-Attention** | Q from decoder, K/V from frozen encoder |
| 15 | **Encoder-Decoder** | Source encoded once; decoder cross-attends at every step |
| 16 | **Architecture Comparison** | Decoder-only won at scale; encoder stays essential |
| 17 | **GPT-2 Internals** | A real model, cracked open |
| 18 | **Autoregressive loop** | Sample next token -> append -> repeat |

### Key insights to keep

- **RoPE** encodes position by rotating Q and K; only the relative gap survives in dot-products
- **Attention is O(n^2)** in sequence length - this is why long-context models are expensive
- **Multi-head attention** lets each head specialise on a different relationship type
- **Residual connections** make depth practical - gradients always have a direct path home
- **Temperature** is the single most intuitive control knob at inference time
- **W_V is a relevance filter** - it extracts the task-specific slice of each token's information
- **The causal triangle means depth compounds** - position n at layer L has processed a chain of enriched representations from all n earlier tokens
- **Encoder = mask removed** - `mask=None` gives every token a full-sentence view from layer 1
- **Cross-attention decouples source and target** - Q from the decoder re-queries a frozen encoder map at every step; no compression bottleneck
- **Decoder-only scales cleanly** - no paired data needed


---

### One Last Bridge: From Two Workspaces to One Conversation

Return to the reversal task one last time. Suppose the source is `[3, 1, 4, 1]` and the answer should be `[1, 4, 1, 3]`.

| Training view | What the model sees | What gets graded |
| --- | --- | --- |
| Encoder-decoder | The encoder keeps `[3, 1, 4, 1]` on one workspace while the decoder builds `[1, 4, 1, 3]` on another. Cross-attention lets the decoder look back at the source. | The decoder's target tokens |
| Decoder-only causal training | The instruction, source, and answer are written on one growing tape. Each position can see only what came before it. | Every next token on the tape |
| Response-masked SFT | The same one-tape layout is used, but the instruction and source are treated as the brief, not the model's work product. | Only the answer tokens |

Nothing magical happened to the learning rule: the model still becomes better at predicting the next token. What changed is the **layout of the conversation** and the **boundary around the work we grade**.

That boundary matters in Riverside's next task. The assistant must read an Aria scene and the instruction `continue in one sentence`, but Riverside does not need to train it to reproduce the brief. The brief stays visible; only the one-sentence response receives the lesson.

This is the bridge into decoder-only fine-tuning: not a new kind of intelligence, just one shared tape and a more intentional answer key.

---

## What's Next - Fine-Tuning a Decoder-Only Assistant

This notebook built the full encoder-decoder architecture used by T5 and BART. The next chapter (`../03-llm-finetuning/`) takes a different path: **decoder-only** models like GPT-2.

| Architecture                | Strength                                                                                     | Weakness at scale                                                                                             |
| --------------------------- | -------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------- |
| Encoder-Decoder (T5, BART)  | Optimal for fixed-format transductions (translation, summarisation, Q→A with known length)   | Two separate stacks to store and serve; encoder parameters add cost without benefit for open-ended generation |
| Decoder-Only (GPT-2, LLaMA) | Single stack; autoregressive pretraining on raw text scales to arbitrary tasks via prompting | No explicit bidirectional context over the source — must include source in the prompt                         |
| Encoder-Only (BERT)         | Best for classification, retrieval, embedding                                                | Cannot generate                                                                                               |

At the scale of GPT-3 (175B parameters), training an encoder-decoder model would have required double the parameter budget for tasks where the encoder's bidirectional attention didn't pay off. Decoder-only models generalised surprisingly well to translation and summarisation via few-shot prompting — removing the case for the extra encoder stack.

→ **Next:** `../03-llm-finetuning/01-llm-finetuning-data-techniques.ipynb` — fine-tuning a decoder-only model on a private corpus under real compute constraints.
